# AC-PFL — Main Experiments (FD001, FD004)

Experiment runs for FedAvg, FedPer, FedProx, Ditto, CFL, and AC-PFL on the FD001 and FD004 C-MAPSS subsets, across multiple seeds, including the FD004 condition-subset (C=3) diagnostic runs.

Core algorithm code lives in [`../src/`](../src/) — this notebook only contains the calls that produced the logged results in [`../results/`](../results/).


In [ ]:
# Core modules live in ../src/ (config.py, utils.py, preprocess.py, run_experiment.py)
# On Kaggle/Colab, copy src/*.py into the working directory first, e.g.:
# import shutil; [shutil.copy(f'../src/{f}', '.') for f in
#     ['config.py', 'utils.py', 'preprocess.py', 'run_experiment.py']]


In [5]:
import os

# Lock the GPUs before TensorFlow wakes up
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ['PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION'] = 'python'

In [ ]:
import itertools
import os
import pandas as pd
import traceback

OUTPUT = '/kaggle/working/results.csv'

def already_completed(output_file, method, dataset, seed):
    if not os.path.exists(output_file):
        return False
    try:
        df = pd.read_csv(output_file)
        return any(
            (df['method'] == method) &
            (df['dataset'] == dataset) &
            (df['seed'] == seed)
        )
    except Exception:
        return False

def save_result(result, output_file):
    exists = os.path.exists(output_file)
    import csv
    with open(output_file, 'a', newline='') as f:
        w = csv.DictWriter(f, fieldnames=result.keys())
        if not exists:
            w.writeheader()
        w.writerow(result)
    print(f"  ✅ Saved: {result['method']} {result['dataset']} "
          f"seed {result['seed']} | "
          f"MAE={result['test_mae']} | NASA={result['nasa_score']}")

from run_experiment import run_simulation, run_cfl, run_acpfl

METHODS = ['fedavg', 'fedprox', 'fedper', 'ditto', 'cfl', 'acpfl']
DATASETS = ['FD001', 'FD002', 'FD003', 'FD004']
SEEDS = [42, 101, 2026]

# Assign methods to accounts to parallelize
# Account 1: fedavg, fedprox
# Account 2: fedper, ditto  
# Account 3: cfl
# Account 4: acpfl

# Change this per account
METHODS_THIS_ACCOUNT = ['fedavg', 'fedprox']  # CHANGE PER ACCOUNT

total = len(METHODS_THIS_ACCOUNT) * len(DATASETS) * len(SEEDS)
completed = 0
skipped = 0
failed = 0

print(f"Total planned runs: {total}")
print(f"Methods: {METHODS_THIS_ACCOUNT}")

for method, dataset, seed in itertools.product(
    METHODS_THIS_ACCOUNT, DATASETS, SEEDS
):
    if already_completed(OUTPUT, method, dataset, seed):
        print(f"  ⏭️  Skip: {method} {dataset} seed {seed}")
        skipped += 1
        continue

    print(f"\n{'='*50}")
    print(f"  Running: {method} | {dataset} | seed {seed}")
    print(f"  Progress: {completed} done, {skipped} skipped, {failed} failed")
    print(f"{'='*50}")

    try:
        if method == 'cfl':
            result = run_cfl(dataset, seed)
        elif method == 'acpfl':
            result = run_acpfl(dataset, seed, alpha=0.5)
        else:
            result = run_simulation(method, dataset, seed)

        save_result(result, OUTPUT)
        completed += 1

    except Exception as e:
        print(f"  ❌ FAILED: {method} {dataset} seed {seed}")
        print(f"  Error: {e}")
        traceback.print_exc()
        failed += 1
        continue

print(f"\n{'='*50}")
print(f"DONE. Completed: {completed} | Skipped: {skipped} | Failed: {failed}")

In [9]:
from run_experiment import run_simulation
print("Checking FedAvg seed 101...")
result = run_simulation('fedavg', 'FD004', 101)
print("FedAvg 101:", result)

Checking FedAvg seed 101...

-- K-Means Clustering Results --
  - Cluster 0 assigned 132 engines.
  - Cluster 1 assigned 117 engines.
---------------------------------
✅ Created sequences: X shape = (11353, 30, 24), y shape = (11353,)
✅ Created sequences: X shape = (3155, 30, 24), y shape = (3155,)
✅ Created sequences: X shape = (11101, 30, 24), y shape = (11101,)
✅ Created sequences: X shape = (3405, 30, 24), y shape = (3405,)
✅ Created sequences: X shape = (10244, 30, 24), y shape = (10244,)
✅ Created sequences: X shape = (2662, 30, 24), y shape = (2662,)
✅ Created sequences: X shape = (9748, 30, 24), y shape = (9748,)
✅ Created sequences: X shape = (2360, 30, 24), y shape = (2360,)


INFO flwr 2026-06-29 12:08:59,416 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-06-29 12:09:08,244	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-06-29 12:09:11,549 | app.py:210 | Flower VCE: Ray initialized with resources: {'CPU': 4.0, 'node:__internal_head__': 1.0, 'GPU': 2.0, 'memory': 18502853018.0, 'object_store_memory': 7929794150.0, 'accelerator_type:T4': 1.0, 'node:172.19.2.2': 1.0}
INFO flwr 2026-06-29 12:09:11,550 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-06-29 12:09:11,651 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-06-29 12:09:11,652 | server.py:89 | Initializing global parameters
INFO flwr 2026-06-29 12:09:11,654 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-06-29 12:09:11,655 | server.py:91 | Evaluating initial parameters
(pid=31983) WARNING: All

  [Round 0] Test MAE: 78.6153 | NASA: 1644382.07


(DefaultActor pid=31984) I0000 00:00:1782734962.017995   31984 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13654 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
(pid=31984) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR [repeated 3x across cluster] (Ray deduplicates logs by default. Set RAY_DEDUP_LOGS=0 to disable log deduplication, or see https://docs.ray.io/en/master/ray-observability/user-guides/configure-logging.html#log-deduplication for more options.)
(pid=31984) E0000 00:00:1782734952.773077   31984 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered [repeated 3x across cluster]
(pid=31984) E0000 00:00:1782734952.797318   31984 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered [repeat

  [Round 1] Test MAE: 34.6112 | NASA: 12939.21


...

  [Round 49] Test MAE: 21.9383 | NASA: 16324.28


DEBUG flwr 2026-06-29 12:58:55,348 | server.py:187 | evaluate_round 49 received 4 results and 0 failures
DEBUG flwr 2026-06-29 12:58:55,349 | server.py:222 | fit_round 50: strategy sampled 4 clients (out of 4)
DEBUG flwr 2026-06-29 12:59:43,637 | server.py:236 | fit_round 50 received 4 results and 0 failures
INFO flwr 2026-06-29 12:59:48,679 | server.py:125 | fit progress: (50, 0.0, {'mae': 22.38461628267842, 'nasa_score': 12846.124463191962}, 3029.6043184459995)
DEBUG flwr 2026-06-29 12:59:48,680 | server.py:173 | evaluate_round 50: strategy sampled 4 clients (out of 4)


  [Round 50] Test MAE: 22.3846 | NASA: 12846.12


DEBUG flwr 2026-06-29 12:59:52,220 | server.py:187 | evaluate_round 50 received 4 results and 0 failures
INFO flwr 2026-06-29 12:59:52,221 | server.py:153 | FL finished in 3033.146371336001
INFO flwr 2026-06-29 12:59:52,222 | app.py:225 | app_fit: losses_distributed [(1, 2110.3554153560954), (2, 582.3708549266322), (3, 503.0775718998567), (4, 505.333132544076), (5, 520.8408760989068), (6, 533.2610897432438), (7, 622.6179469613662), (8, 540.0756214677392), (9, 621.449569562498), (10, 564.5430481452395), (11, 584.4884499688363), (12, 603.0074242084802), (13, 569.4978849437031), (14, 590.3279573193199), (15, 598.5299924133437), (16, 587.7251478144798), (17, 637.1615664719753), (18, 581.8152727520006), (19, 613.297999001768), (20, 606.015488606293), (21, 678.1334789699686), (22, 633.8578895196007), (23, 635.6229544527439), (24, 641.7534628940399), (25, 654.1987102694965), (26, 625.7253406638619), (27, 674.200442593943), (28, 630.1799470074455), (29, 630.1054458947528), (30, 620.75927945168

FedAvg 101: {'method': 'fedavg', 'dataset': 'FD004', 'seed': 101, 'test_mae': 22.3846, 'nasa_score': 12846.12, 'comm_kb': 28900.78}


In [8]:
from run_experiment import run_acpfl
print("Checking AC-PFL seed 101...")
result = run_acpfl('FD004', 101, alpha=0.5)
print("AC-PFL 101:", result)

Checking AC-PFL seed 101...

-- K-Means Clustering Results --
  - Cluster 0 assigned 132 engines.
  - Cluster 1 assigned 117 engines.
---------------------------------
✅ Created sequences: X shape = (11353, 30, 24), y shape = (11353,)
✅ Created sequences: X shape = (3155, 30, 24), y shape = (3155,)
✅ Created sequences: X shape = (11101, 30, 24), y shape = (11101,)
✅ Created sequences: X shape = (3405, 30, 24), y shape = (3405,)
✅ Created sequences: X shape = (10244, 30, 24), y shape = (10244,)
✅ Created sequences: X shape = (2662, 30, 24), y shape = (2662,)
✅ Created sequences: X shape = (9748, 30, 24), y shape = (9748,)
✅ Created sequences: X shape = (2360, 30, 24), y shape = (2360,)


I0000 00:00:1782728957.816347     112 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1782728957.822175     112 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1782728965.445161     165 cuda_dnn.cc:529] Loaded cuDNN version 91002


  [Round 1] Val NASA per client: [76419.0, 104342.7, 102440.1, 37814.8]
  [Round 2] Val NASA per client: [60500.2, 131413.1, 38912.4, 30253.3]
  [Round 3] Val NASA per client: [60125.7, 73950.0, 32040.1, 33713.2]
  [Round 4] Val NASA per client: [129263.5, 65683.5, 37613.8, 30054.7]
  [Round 5] Re-clustered (α=1.0). Changes: {1: (0, 1), 2: (1, 0)}
  [Round 5] Assignments: {'0': 0, '1': 1, '2': 0, '3': 1}
  [Round 5] Cluster 0 val NASA: 69654.28
  [Round 5] Cluster 1 val NASA: 48264.89
  [Round 5] Val NASA per client: [87740.9, 63574.2, 51567.7, 32955.6]
  [Round 6] Val NASA per client: [61567.2, 74104.3, 31157.7, 29151.1]
  [Round 7] Val NASA per client: [133787.5, 82914.6, 30320.7, 36528.0]
  [Round 8] Val NASA per client: [48399.7, 80176.4, 51081.3, 30661.1]
  [Round 9] Val NASA per client: [60160.4, 92043.0, 38400.3, 29203.1]
  [Round 10] Re-clustered (α=0.5). Changes: {1: (1, 0), 2: (0, 1)}
  [Round 10] Assignments: {'0': 0, '1': 0, '2': 1, '3': 1}
  [Round 10] Cluster 0 val NASA: 

In [6]:
from run_experiment import run_acpfl
print("Checking AC-PFL seed 42...")
result = run_acpfl('FD004', 42, alpha=0.5)
print("AC-PFL 42:", result)

E0000 00:00:1782750267.857125     112 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782750267.981688     112 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782750268.942533     112 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782750268.942584     112 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782750268.942587     112 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782750268.942589     112 computation_placer.cc:177] computation placer already registered. Please check linka

Checking AC-PFL seed 42...

-- K-Means Clustering Results --
  - Cluster 0 assigned 132 engines.
  - Cluster 1 assigned 117 engines.
---------------------------------
✅ Created sequences: X shape = (12060, 30, 24), y shape = (12060,)
✅ Created sequences: X shape = (2448, 30, 24), y shape = (2448,)
✅ Created sequences: X shape = (11660, 30, 24), y shape = (11660,)
✅ Created sequences: X shape = (2846, 30, 24), y shape = (2846,)
✅ Created sequences: X shape = (10505, 30, 24), y shape = (10505,)
✅ Created sequences: X shape = (2401, 30, 24), y shape = (2401,)
✅ Created sequences: X shape = (9866, 30, 24), y shape = (9866,)
✅ Created sequences: X shape = (2242, 30, 24), y shape = (2242,)


I0000 00:00:1782750339.389640     112 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1782750339.395700     112 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1782750347.340774     180 cuda_dnn.cc:529] Loaded cuDNN version 91002


  [Round 1] Val NASA per client: [139898.1, 49667.5, 41624.7, 32652.3]
  [Round 2] Val NASA per client: [143416.2, 95197.9, 41664.9, 17370.7]
  [Round 3] Val NASA per client: [176482.2, 63069.0, 32794.7, 12313.4]
  [Round 4] Val NASA per client: [307796.9, 93731.4, 24326.9, 12635.3]
  ⚠️ Spectral produced invalid clusters {0: 1, 1: 3}. Falling back to KMeans.
  KMeans fallback result: {1: 1, 0: 3}
  [Round 5] Re-clustered (α=1.0). Changes: {0: (0, 1), 2: (1, 0), 3: (1, 0)}
  [Round 5] Assignments: {'0': 1, '1': 0, '2': 0, '3': 0}
  [Round 5] Cluster 0 val NASA: 48300.18
  [Round 5] Cluster 1 val NASA: 209179.38
  [Round 5] Val NASA per client: [209179.4, 99070.2, 30381.2, 15449.1]
  [Round 6] Val NASA per client: [39358.2, 126635.7, 27570.5, 25448.9]
  [Round 7] Val NASA per client: [120220.2, 87195.0, 27480.0, 18245.4]
  [Round 8] Val NASA per client: [68654.7, 71723.1, 34401.3, 19520.5]
  [Round 9] Val NASA per client: [55398.2, 136375.9, 30807.5, 25683.1]
  [Round 10] Re-clustered (

In [7]:
from run_experiment import run_simulation
print("Checking FedAvg seed 42...")
result = run_simulation('fedavg', 'FD004', 42)
print("FedAvg 42:", result)

Checking FedAvg seed 42...

-- K-Means Clustering Results --
  - Cluster 0 assigned 132 engines.
  - Cluster 1 assigned 117 engines.
---------------------------------
✅ Created sequences: X shape = (12060, 30, 24), y shape = (12060,)
✅ Created sequences: X shape = (2448, 30, 24), y shape = (2448,)
✅ Created sequences: X shape = (11660, 30, 24), y shape = (11660,)
✅ Created sequences: X shape = (2846, 30, 24), y shape = (2846,)
✅ Created sequences: X shape = (10505, 30, 24), y shape = (10505,)
✅ Created sequences: X shape = (2401, 30, 24), y shape = (2401,)
✅ Created sequences: X shape = (9866, 30, 24), y shape = (9866,)
✅ Created sequences: X shape = (2242, 30, 24), y shape = (2242,)


INFO flwr 2026-06-29 18:15:39,074 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-06-29 18:15:48,409	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-06-29 18:15:52,232 | app.py:210 | Flower VCE: Ray initialized with resources: {'node:172.19.2.2': 1.0, 'memory': 18396964455.0, 'node:__internal_head__': 1.0, 'GPU': 2.0, 'accelerator_type:T4': 1.0, 'object_store_memory': 7884413337.0, 'CPU': 4.0}
INFO flwr 2026-06-29 18:15:52,233 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-06-29 18:15:52,333 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-06-29 18:15:52,334 | server.py:89 | Initializing global parameters
INFO flwr 2026-06-29 18:15:52,336 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-06-29 18:15:52,336 | server.py:91 | Evaluating initial parameters
(pid=33157) WARNING: All

  [Round 0] Test MAE: 80.3084 | NASA: 1874047.15


(DefaultActor pid=33157) I0000 00:00:1782756963.311607   33157 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13374 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
(pid=33156) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR [repeated 3x across cluster] (Ray deduplicates logs by default. Set RAY_DEDUP_LOGS=0 to disable log deduplication, or see https://docs.ray.io/en/master/ray-observability/user-guides/configure-logging.html#log-deduplication for more options.)
(pid=33156) E0000 00:00:1782756953.511946   33156 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered [repeated 3x across cluster]
(pid=33156) E0000 00:00:1782756953.541538   33156 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered [repeat

  [Round 1] Test MAE: 39.0170 | NASA: 73795.66


...

  [Round 49] Test MAE: 21.1101 | NASA: 17697.67


DEBUG flwr 2026-06-29 19:10:19,004 | server.py:187 | evaluate_round 49 received 4 results and 0 failures
DEBUG flwr 2026-06-29 19:10:19,005 | server.py:222 | fit_round 50: strategy sampled 4 clients (out of 4)
DEBUG flwr 2026-06-29 19:11:15,365 | server.py:236 | fit_round 50 received 4 results and 0 failures
INFO flwr 2026-06-29 19:11:20,694 | server.py:125 | fit progress: (50, 0.0, {'mae': 21.556122179954283, 'nasa_score': 31440.990196730054}, 3320.5397188999996)
DEBUG flwr 2026-06-29 19:11:20,695 | server.py:173 | evaluate_round 50: strategy sampled 4 clients (out of 4)


  [Round 50] Test MAE: 21.5561 | NASA: 31440.99


DEBUG flwr 2026-06-29 19:11:23,265 | server.py:187 | evaluate_round 50 received 4 results and 0 failures
INFO flwr 2026-06-29 19:11:23,266 | server.py:153 | FL finished in 3323.1120386559996
INFO flwr 2026-06-29 19:11:23,268 | app.py:225 | app_fit: losses_distributed [(1, 1888.2686313791535), (2, 740.4444458646602), (3, 712.6677854725493), (4, 574.3667632697321), (5, 574.737801039616), (6, 561.780703564291), (7, 568.9570834250161), (8, 589.1928111457652), (9, 632.4505906743735), (10, 607.16072449056), (11, 650.7888930670941), (12, 645.4695448451711), (13, 658.5004925439448), (14, 691.6299695958071), (15, 691.9236625432608), (16, 743.6538397409048), (17, 725.5946467260342), (18, 724.0221167156799), (19, 696.5653986570472), (20, 715.1448718172043), (21, 719.6221619604387), (22, 723.6485659459281), (23, 724.2454473006602), (24, 741.4719130301171), (25, 718.7479872955479), (26, 725.2752653853063), (27, 762.5117739316287), (28, 741.4240322061212), (29, 786.57879630687), (30, 740.56120001320

FedAvg 42: {'method': 'fedavg', 'dataset': 'FD004', 'seed': 42, 'test_mae': 21.5561, 'nasa_score': 31440.99, 'comm_kb': 28900.78}


In [6]:
from run_experiment import run_acpfl
print("Checking AC-PFL seed 2026...")
result = run_acpfl('FD004', 2026, alpha=0.5)
print("AC-PFL 2026:", result)

E0000 00:00:1782806943.769476     114 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782806943.836301     114 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782806944.373266     114 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782806944.373320     114 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782806944.373323     114 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782806944.373325     114 computation_placer.cc:177] computation placer already registered. Please check linka

Checking AC-PFL seed 2026...

-- K-Means Clustering Results --
  - Cluster 0 assigned 132 engines.
  - Cluster 1 assigned 117 engines.
---------------------------------
✅ Created sequences: X shape = (11559, 30, 24), y shape = (11559,)
✅ Created sequences: X shape = (2949, 30, 24), y shape = (2949,)
✅ Created sequences: X shape = (11576, 30, 24), y shape = (11576,)
✅ Created sequences: X shape = (2930, 30, 24), y shape = (2930,)
✅ Created sequences: X shape = (10377, 30, 24), y shape = (10377,)
✅ Created sequences: X shape = (2529, 30, 24), y shape = (2529,)
✅ Created sequences: X shape = (9810, 30, 24), y shape = (9810,)
✅ Created sequences: X shape = (2298, 30, 24), y shape = (2298,)


I0000 00:00:1782807007.984322     114 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1782807007.990197     114 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1782807015.277449     164 cuda_dnn.cc:529] Loaded cuDNN version 91002


  [Round 1] Val NASA per client: [248610.6, 62376.2, 50338.8, 59521.0]
  [Round 2] Val NASA per client: [21423.7, 83794.0, 42785.2, 89356.1]
  [Round 3] Val NASA per client: [26814.3, 79425.2, 36295.0, 47739.7]
  [Round 4] Val NASA per client: [19332.5, 55592.1, 33722.5, 49475.0]
  [Round 5] Re-clustered (α=1.0). Changes: {1: (0, 1), 3: (1, 0)}
  [Round 5] Assignments: {'0': 0, '1': 1, '2': 1, '3': 0}
  [Round 5] Cluster 0 val NASA: 36741.42
  [Round 5] Cluster 1 val NASA: 51691.86
  [Round 5] Val NASA per client: [20855.3, 60652.8, 42730.9, 52627.5]
  [Round 6] Val NASA per client: [24193.2, 70768.5, 35645.9, 31192.6]
  [Round 7] Val NASA per client: [27469.9, 64258.1, 40833.1, 49291.4]
  [Round 8] Val NASA per client: [19390.7, 90996.0, 35130.0, 36668.9]
  [Round 9] Val NASA per client: [15231.5, 60834.7, 45032.8, 41670.0]
  [Round 10] Re-clustered (α=0.5). Changes: {2: (1, 0), 3: (0, 1)}
  [Round 10] Assignments: {'0': 0, '1': 1, '2': 0, '3': 1}
  [Round 10] Cluster 0 val NASA: 5162

In [6]:
from run_experiment import run_simulation
print("Checking FedAvg seed 202...")
result = run_simulation('fedavg', 'FD004', 202)
print("FedAvg 202:", result)

E0000 00:00:1783061413.014009     112 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1783061413.081289     112 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1783061413.597749     112 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783061413.597800     112 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783061413.597802     112 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783061413.597805     112 computation_placer.cc:177] computation placer already registered. Please check linka

Checking FedAvg seed 202...

-- K-Means Clustering Results --
  - Cluster 0 assigned 132 engines.
  - Cluster 1 assigned 117 engines.
---------------------------------
✅ Created sequences: X shape = (11720, 30, 24), y shape = (11720,)
✅ Created sequences: X shape = (2788, 30, 24), y shape = (2788,)
✅ Created sequences: X shape = (11430, 30, 24), y shape = (11430,)
✅ Created sequences: X shape = (3076, 30, 24), y shape = (3076,)
✅ Created sequences: X shape = (10465, 30, 24), y shape = (10465,)
✅ Created sequences: X shape = (2441, 30, 24), y shape = (2441,)
✅ Created sequences: X shape = (9809, 30, 24), y shape = (9809,)
✅ Created sequences: X shape = (2299, 30, 24), y shape = (2299,)


I0000 00:00:1783061475.680673     112 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1783061475.686557     112 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
INFO flwr 2026-07-03 06:51:17,556 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-03 06:51:25,606	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-03 06:51:28,942 | app.py:210 | Flower VCE: Ray initialized with resources: {'GPU': 2.0, 'accelerator_type:T4': 1.0, 'memory': 21526421504.0, 'object_store_memory': 9225609216.0, 'node:172.19.2.2': 1.0, 'CPU': 4.0, 'node:__internal_head__': 1.0}
INFO flwr 2026-07-03 06:51:28,943 | app.py:224 | Flower VCE: Resources for each Virtu

  [Round 0] Test MAE: 79.5096 | NASA: 1762216.56


(DefaultActor pid=362) I0000 00:00:1783061499.903469     362 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13634 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
(pid=361) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR [repeated 3x across cluster] (Ray deduplicates logs by default. Set RAY_DEDUP_LOGS=0 to disable log deduplication, or see https://docs.ray.io/en/master/ray-observability/user-guides/configure-logging.html#log-deduplication for more options.)
(pid=361) E0000 00:00:1783061490.363995     361 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered [repeated 3x across cluster]
(pid=361) E0000 00:00:1783061490.377519     361 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered [repeated 3x ac

...

  [Round 49] Test MAE: 22.3185 | NASA: 70375.75


DEBUG flwr 2026-07-03 07:41:29,107 | server.py:187 | evaluate_round 49 received 4 results and 0 failures
DEBUG flwr 2026-07-03 07:41:29,108 | server.py:222 | fit_round 50: strategy sampled 4 clients (out of 4)
DEBUG flwr 2026-07-03 07:42:30,833 | server.py:236 | fit_round 50 received 4 results and 0 failures
INFO flwr 2026-07-03 07:42:32,150 | server.py:125 | fit progress: (50, 0.0, {'mae': 22.7314583716854, 'nasa_score': 94065.3112069071}, 3059.460326807)
DEBUG flwr 2026-07-03 07:42:32,151 | server.py:173 | evaluate_round 50: strategy sampled 4 clients (out of 4)


  [Round 50] Test MAE: 22.7315 | NASA: 94065.31


DEBUG flwr 2026-07-03 07:42:34,537 | server.py:187 | evaluate_round 50 received 4 results and 0 failures
INFO flwr 2026-07-03 07:42:34,538 | server.py:153 | FL finished in 3061.848423547
INFO flwr 2026-07-03 07:42:34,539 | app.py:225 | app_fit: losses_distributed [(1, 1978.5757001936547), (2, 707.5090152448369), (3, 606.6025778512512), (4, 593.8671045436449), (5, 554.3076220368134), (6, 555.7166692330135), (7, 568.347766365388), (8, 585.4292547397908), (9, 598.8238107889205), (10, 678.0012805122827), (11, 633.0437550985422), (12, 642.21263172059), (13, 676.6232511390159), (14, 731.246148430685), (15, 701.0002635148551), (16, 707.7408619826356), (17, 701.6863767781468), (18, 742.1739606594689), (19, 740.4893655334496), (20, 744.5630167900324), (21, 752.3963596569912), (22, 774.5815005250266), (23, 739.188650343473), (24, 750.283804180846), (25, 749.726894964321), (26, 759.6116933747086), (27, 747.988471250991), (28, 734.5217870412256), (29, 764.4190314144334), (30, 748.9624733883495), (

FedAvg 202: {'method': 'fedavg', 'dataset': 'FD004', 'seed': 202, 'test_mae': 22.7315, 'nasa_score': 94065.31, 'comm_kb': 28900.78}


In [6]:
from run_experiment import run_simulation
print("Checking FedAvg seed 303...")
result = run_simulation('fedavg', 'FD004', 303)
print("FedAvg 303:", result)

E0000 00:00:1783075039.478013     114 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1783075039.542805     114 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1783075040.050049     114 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783075040.050106     114 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783075040.050109     114 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783075040.050112     114 computation_placer.cc:177] computation placer already registered. Please check linka

Checking FedAvg seed 303...

-- K-Means Clustering Results --
  - Cluster 0 assigned 132 engines.
  - Cluster 1 assigned 117 engines.
---------------------------------
✅ Created sequences: X shape = (11380, 30, 24), y shape = (11380,)
✅ Created sequences: X shape = (3128, 30, 24), y shape = (3128,)
✅ Created sequences: X shape = (10903, 30, 24), y shape = (10903,)
✅ Created sequences: X shape = (3603, 30, 24), y shape = (3603,)
✅ Created sequences: X shape = (10533, 30, 24), y shape = (10533,)
✅ Created sequences: X shape = (2373, 30, 24), y shape = (2373,)
✅ Created sequences: X shape = (9746, 30, 24), y shape = (9746,)
✅ Created sequences: X shape = (2362, 30, 24), y shape = (2362,)


I0000 00:00:1783075103.775434     114 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1783075103.781362     114 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
INFO flwr 2026-07-03 10:38:25,688 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-03 10:38:33,732	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-03 10:38:37,522 | app.py:210 | Flower VCE: Ray initialized with resources: {'node:__internal_head__': 1.0, 'object_store_memory': 9226535731.0, 'node:172.19.2.2': 1.0, 'GPU': 2.0, 'memory': 21528583373.0, 'CPU': 4.0, 'accelerator_type:T4': 1.0}
INFO flwr 2026-07-03 10:38:37,522 | app.py:224 | Flower VCE: Resources for each Virtu

  [Round 0] Test MAE: 78.8251 | NASA: 1673564.37


(DefaultActor pid=363) I0000 00:00:1783075127.845864     363 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13606 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
(pid=361) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR [repeated 3x across cluster] (Ray deduplicates logs by default. Set RAY_DEDUP_LOGS=0 to disable log deduplication, or see https://docs.ray.io/en/master/ray-observability/user-guides/configure-logging.html#log-deduplication for more options.)
(pid=361) E0000 00:00:1783075118.737036     361 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered [repeated 3x across cluster]
(pid=362) E0000 00:00:1783075118.691712     362 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered [repeated 3x ac

...

  [Round 49] Test MAE: 22.7799 | NASA: 67638.28


DEBUG flwr 2026-07-03 11:31:37,649 | server.py:187 | evaluate_round 49 received 4 results and 0 failures
DEBUG flwr 2026-07-03 11:31:37,650 | server.py:222 | fit_round 50: strategy sampled 4 clients (out of 4)
DEBUG flwr 2026-07-03 11:32:33,127 | server.py:236 | fit_round 50 received 4 results and 0 failures
INFO flwr 2026-07-03 11:32:34,571 | server.py:125 | fit progress: (50, 0.0, {'mae': 22.512658088437973, 'nasa_score': 63331.47683455485}, 3233.285020669)
DEBUG flwr 2026-07-03 11:32:34,573 | server.py:173 | evaluate_round 50: strategy sampled 4 clients (out of 4)


  [Round 50] Test MAE: 22.5127 | NASA: 63331.48


DEBUG flwr 2026-07-03 11:32:37,269 | server.py:187 | evaluate_round 50 received 4 results and 0 failures
INFO flwr 2026-07-03 11:32:37,270 | server.py:153 | FL finished in 3235.983455312
INFO flwr 2026-07-03 11:32:37,270 | app.py:225 | app_fit: losses_distributed [(1, 1437.0267576826154), (2, 656.9371791582092), (3, 748.0285232360297), (4, 662.8575606653043), (5, 682.9961117901312), (6, 688.287222859083), (7, 632.6345247314922), (8, 692.0194791577238), (9, 655.4494270392577), (10, 753.8573837879127), (11, 723.7580004921139), (12, 735.724271499152), (13, 677.1994500458812), (14, 846.710310886236), (15, 763.1440502454864), (16, 718.7075967700532), (17, 756.329955666098), (18, 716.749109571275), (19, 759.7661672206568), (20, 810.4546228685284), (21, 763.9815144654491), (22, 723.1131362475879), (23, 789.8011531141152), (24, 764.7861593164284), (25, 734.2372991301388), (26, 765.9100751572414), (27, 736.6044332017537), (28, 746.9847001269208), (29, 827.6321777365042), (30, 770.3281439397424)

FedAvg 303: {'method': 'fedavg', 'dataset': 'FD004', 'seed': 303, 'test_mae': 22.5127, 'nasa_score': 63331.48, 'comm_kb': 28900.78}


In [7]:
from run_experiment import run_simulation
print("Checking FedPer seed 42...")
result = run_simulation('fedper', 'FD004', 42)
print("FedPer 42:", result)

Checking FedPer seed 42...

-- K-Means Clustering Results --
  - Cluster 0 assigned 132 engines.
  - Cluster 1 assigned 117 engines.
---------------------------------
✅ Created sequences: X shape = (12060, 30, 24), y shape = (12060,)
✅ Created sequences: X shape = (2448, 30, 24), y shape = (2448,)
✅ Created sequences: X shape = (11660, 30, 24), y shape = (11660,)
✅ Created sequences: X shape = (2846, 30, 24), y shape = (2846,)
✅ Created sequences: X shape = (10505, 30, 24), y shape = (10505,)
✅ Created sequences: X shape = (2401, 30, 24), y shape = (2401,)
✅ Created sequences: X shape = (9866, 30, 24), y shape = (9866,)
✅ Created sequences: X shape = (2242, 30, 24), y shape = (2242,)


INFO flwr 2026-07-03 11:33:47,338 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-03 11:33:59,767	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-03 11:34:03,460 | app.py:210 | Flower VCE: Ray initialized with resources: {'accelerator_type:T4': 1.0, 'node:__internal_head__': 1.0, 'memory': 15245232128.0, 'node:172.19.2.2': 1.0, 'GPU': 2.0, 'CPU': 4.0, 'object_store_memory': 6533670912.0}
INFO flwr 2026-07-03 11:34:03,461 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-07-03 11:34:03,500 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-07-03 11:34:03,503 | server.py:89 | Initializing global parameters
INFO flwr 2026-07-03 11:34:03,506 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-07-03 11:34:03,509 | server.py:91 | Evaluating initial parameters
INFO flwr 2026-07-03 11:

FedPer 42: {'method': 'fedper', 'dataset': 'FD004', 'seed': 42, 'test_mae': 21.8379, 'nasa_score': 59686.66, 'comm_kb': 27650.0}


In [6]:
from run_experiment import run_simulation
print("Checking FedPer seed 101...")
result = run_simulation('fedper', 'FD004', 101)
print("FedPer 101:", result)

E0000 00:00:1783509633.876888     113 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1783509633.953166     113 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1783509634.562049     113 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783509634.562092     113 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783509634.562094     113 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783509634.562097     113 computation_placer.cc:177] computation placer already registered. Please check linka

Checking FedPer seed 101...

-- K-Means Clustering Results --
  - Cluster 0 assigned 132 engines.
  - Cluster 1 assigned 117 engines.
---------------------------------
✅ Created sequences: X shape = (11353, 30, 24), y shape = (11353,)
✅ Created sequences: X shape = (3155, 30, 24), y shape = (3155,)
✅ Created sequences: X shape = (11101, 30, 24), y shape = (11101,)
✅ Created sequences: X shape = (3405, 30, 24), y shape = (3405,)
✅ Created sequences: X shape = (10244, 30, 24), y shape = (10244,)
✅ Created sequences: X shape = (2662, 30, 24), y shape = (2662,)
✅ Created sequences: X shape = (9748, 30, 24), y shape = (9748,)
✅ Created sequences: X shape = (2360, 30, 24), y shape = (2360,)


I0000 00:00:1783509699.617368     113 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1783509699.623547     113 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
INFO flwr 2026-07-08 11:21:41,594 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-08 11:21:49,653	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-08 11:21:53,188 | app.py:210 | Flower VCE: Ray initialized with resources: {'accelerator_type:T4': 1.0, 'CPU': 4.0, 'node:172.19.2.2': 1.0, 'node:__internal_head__': 1.0, 'GPU': 2.0, 'object_store_memory': 9226522214.0, 'memory': 21528551834.0}
INFO flwr 2026-07-08 11:21:53,189 | app.py:224 | Flower VCE: Resources for each Virtu

FedPer 101: {'method': 'fedper', 'dataset': 'FD004', 'seed': 101, 'test_mae': 21.5991, 'nasa_score': 63286.74, 'comm_kb': 27650.0}


In [7]:
from run_experiment import run_simulation
print("Checking FedPer seed 202...")
result = run_simulation('fedper', 'FD004', 202)
print("FedPer 202:", result)

Checking FedPer seed 202...

-- K-Means Clustering Results --
  - Cluster 0 assigned 132 engines.
  - Cluster 1 assigned 117 engines.
---------------------------------
✅ Created sequences: X shape = (11720, 30, 24), y shape = (11720,)
✅ Created sequences: X shape = (2788, 30, 24), y shape = (2788,)
✅ Created sequences: X shape = (11430, 30, 24), y shape = (11430,)
✅ Created sequences: X shape = (3076, 30, 24), y shape = (3076,)
✅ Created sequences: X shape = (10465, 30, 24), y shape = (10465,)
✅ Created sequences: X shape = (2441, 30, 24), y shape = (2441,)
✅ Created sequences: X shape = (9809, 30, 24), y shape = (9809,)
✅ Created sequences: X shape = (2299, 30, 24), y shape = (2299,)


INFO flwr 2026-07-08 12:10:22,505 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-08 12:10:35,484	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-08 12:10:39,226 | app.py:210 | Flower VCE: Ray initialized with resources: {'memory': 15161320653.0, 'GPU': 2.0, 'CPU': 4.0, 'node:172.19.2.2': 1.0, 'object_store_memory': 6497708851.0, 'node:__internal_head__': 1.0, 'accelerator_type:T4': 1.0}
INFO flwr 2026-07-08 12:10:39,228 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-07-08 12:10:39,260 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-07-08 12:10:39,262 | server.py:89 | Initializing global parameters
INFO flwr 2026-07-08 12:10:39,264 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-07-08 12:10:39,266 | server.py:91 | Evaluating initial parameters
INFO flwr 2026-07-08 12:

FedPer 202: {'method': 'fedper', 'dataset': 'FD004', 'seed': 202, 'test_mae': 21.2135, 'nasa_score': 15891.62, 'comm_kb': 27650.0}


In [8]:
from run_experiment import run_simulation
print("Checking FedPer seed 303...")
result = run_simulation('fedper', 'FD004', 303)
print("FedPer 303:", result)

Checking FedPer seed 303...

-- K-Means Clustering Results --
  - Cluster 0 assigned 132 engines.
  - Cluster 1 assigned 117 engines.
---------------------------------
✅ Created sequences: X shape = (11380, 30, 24), y shape = (11380,)
✅ Created sequences: X shape = (3128, 30, 24), y shape = (3128,)
✅ Created sequences: X shape = (10903, 30, 24), y shape = (10903,)
✅ Created sequences: X shape = (3603, 30, 24), y shape = (3603,)
✅ Created sequences: X shape = (10533, 30, 24), y shape = (10533,)
✅ Created sequences: X shape = (2373, 30, 24), y shape = (2373,)
✅ Created sequences: X shape = (9746, 30, 24), y shape = (9746,)
✅ Created sequences: X shape = (2362, 30, 24), y shape = (2362,)


INFO flwr 2026-07-08 13:00:45,708 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-08 13:00:58,995	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-08 13:01:02,880 | app.py:210 | Flower VCE: Ray initialized with resources: {'CPU': 4.0, 'memory': 15100189082.0, 'node:__internal_head__': 1.0, 'GPU': 2.0, 'object_store_memory': 6471509606.0, 'accelerator_type:T4': 1.0, 'node:172.19.2.2': 1.0}
INFO flwr 2026-07-08 13:01:02,881 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-07-08 13:01:02,911 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-07-08 13:01:02,912 | server.py:89 | Initializing global parameters
INFO flwr 2026-07-08 13:01:02,913 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-07-08 13:01:02,915 | server.py:91 | Evaluating initial parameters
INFO flwr 2026-07-08 13:

FedPer 303: {'method': 'fedper', 'dataset': 'FD004', 'seed': 303, 'test_mae': 23.4532, 'nasa_score': 168380.87, 'comm_kb': 27650.0}


In [9]:
from run_experiment import run_simulation
print("Checking FedPer seed 2026...")
result = run_simulation('fedper', 'FD004', 2026)
print("FedPer 2026:", result)

Checking FedPer seed 2026...

-- K-Means Clustering Results --
  - Cluster 0 assigned 132 engines.
  - Cluster 1 assigned 117 engines.
---------------------------------
✅ Created sequences: X shape = (11559, 30, 24), y shape = (11559,)
✅ Created sequences: X shape = (2949, 30, 24), y shape = (2949,)
✅ Created sequences: X shape = (11576, 30, 24), y shape = (11576,)
✅ Created sequences: X shape = (2930, 30, 24), y shape = (2930,)
✅ Created sequences: X shape = (10377, 30, 24), y shape = (10377,)
✅ Created sequences: X shape = (2529, 30, 24), y shape = (2529,)
✅ Created sequences: X shape = (9810, 30, 24), y shape = (9810,)
✅ Created sequences: X shape = (2298, 30, 24), y shape = (2298,)


INFO flwr 2026-07-08 13:53:49,557 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-08 13:53:57,376	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-08 13:54:01,102 | app.py:210 | Flower VCE: Ray initialized with resources: {'GPU': 2.0, 'node:172.19.2.2': 1.0, 'memory': 21348821402.0, 'node:__internal_head__': 1.0, 'accelerator_type:T4': 1.0, 'object_store_memory': 9149494886.0, 'CPU': 4.0}
INFO flwr 2026-07-08 13:54:01,103 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-07-08 13:54:01,122 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-07-08 13:54:01,123 | server.py:89 | Initializing global parameters
INFO flwr 2026-07-08 13:54:01,125 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-07-08 13:54:01,126 | server.py:91 | Evaluating initial parameters
INFO flwr 2026-07-08 13:

FedPer 2026: {'method': 'fedper', 'dataset': 'FD004', 'seed': 2026, 'test_mae': 22.5931, 'nasa_score': 41220.06, 'comm_kb': 27650.0}


In [10]:
from run_experiment import run_simulation, run_cfl
print("Running CFL K=2...")
cfl_result = run_cfl('FD004', 42)
print("CFL:", cfl_result)

Running CFL K=2...

-- K-Means Clustering Results --
  - Cluster 0 assigned 132 engines.
  - Cluster 1 assigned 117 engines.
---------------------------------
✅ Created sequences: X shape = (12060, 30, 24), y shape = (12060,)
✅ Created sequences: X shape = (2448, 30, 24), y shape = (2448,)
✅ Created sequences: X shape = (11660, 30, 24), y shape = (11660,)
✅ Created sequences: X shape = (2846, 30, 24), y shape = (2846,)
✅ Created sequences: X shape = (10505, 30, 24), y shape = (10505,)
✅ Created sequences: X shape = (2401, 30, 24), y shape = (2401,)
✅ Created sequences: X shape = (9866, 30, 24), y shape = (9866,)
✅ Created sequences: X shape = (2242, 30, 24), y shape = (2242,)
  [Round 1] Global MAE: 39.9858
  [Round 2] Global MAE: 19.3225
  [Round 3] Global MAE: 21.2242
  [Round 4] Global MAE: 18.8952
  [Round 5] CFL one-shot clustering: {0: 0, 1: 1, 2: 0, 3: 1}
  [Round 5] Cluster MAEs: [18.57, 18.57] | Avg: 18.5726
  [Round 6] Cluster MAEs: [18.74, 19.81] | Avg: 19.2746
  [Round 7] C

In [12]:
from run_experiment import run_simulation, run_cfl
print("Running CFL K=2...")
cfl_result = run_cfl('FD004', 101)
print("CFL:", cfl_result)

Running CFL K=2...

-- K-Means Clustering Results --
  - Cluster 0 assigned 132 engines.
  - Cluster 1 assigned 117 engines.
---------------------------------
✅ Created sequences: X shape = (11353, 30, 24), y shape = (11353,)
✅ Created sequences: X shape = (3155, 30, 24), y shape = (3155,)
✅ Created sequences: X shape = (11101, 30, 24), y shape = (11101,)
✅ Created sequences: X shape = (3405, 30, 24), y shape = (3405,)
✅ Created sequences: X shape = (10244, 30, 24), y shape = (10244,)
✅ Created sequences: X shape = (2662, 30, 24), y shape = (2662,)
✅ Created sequences: X shape = (9748, 30, 24), y shape = (9748,)
✅ Created sequences: X shape = (2360, 30, 24), y shape = (2360,)
  [Round 1] Global MAE: 39.8727
  [Round 2] Global MAE: 20.844
  [Round 3] Global MAE: 20.5339
  [Round 4] Global MAE: 19.87
  [Round 5] CFL one-shot clustering: {0: 0, 1: 0, 2: 0, 3: 1}
  [Round 5] Cluster MAEs: [19.5, 19.5] | Avg: 19.4973
  [Round 6] Cluster MAEs: [19.86, 23.77] | Avg: 21.8147
  [Round 7] Cluste

In [13]:
from run_experiment import run_simulation, run_cfl
print("Running CFL K=2...")
cfl_result = run_cfl('FD004', 202)
print("CFL:", cfl_result)

Running CFL K=2...

-- K-Means Clustering Results --
  - Cluster 0 assigned 132 engines.
  - Cluster 1 assigned 117 engines.
---------------------------------
✅ Created sequences: X shape = (11720, 30, 24), y shape = (11720,)
✅ Created sequences: X shape = (2788, 30, 24), y shape = (2788,)
✅ Created sequences: X shape = (11430, 30, 24), y shape = (11430,)
✅ Created sequences: X shape = (3076, 30, 24), y shape = (3076,)
✅ Created sequences: X shape = (10465, 30, 24), y shape = (10465,)
✅ Created sequences: X shape = (2441, 30, 24), y shape = (2441,)
✅ Created sequences: X shape = (9809, 30, 24), y shape = (9809,)
✅ Created sequences: X shape = (2299, 30, 24), y shape = (2299,)
  [Round 1] Global MAE: 39.3373
  [Round 2] Global MAE: 19.6833
  [Round 3] Global MAE: 19.2368
  [Round 4] Global MAE: 18.4973
  [Round 5] CFL one-shot clustering: {0: 0, 1: 1, 2: 0, 3: 1}
  [Round 5] Cluster MAEs: [18.97, 18.97] | Avg: 18.9656
  [Round 6] Cluster MAEs: [20.73, 21.28] | Avg: 21.0096
  [Round 7] C

In [6]:
from run_experiment import run_simulation, run_cfl
print("Running CFL K=2...")
cfl_result = run_cfl('FD004', 303)
print("CFL:", cfl_result)

E0000 00:00:1783574108.824822     114 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1783574108.884111     114 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1783574109.403391     114 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783574109.403438     114 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783574109.403442     114 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783574109.403445     114 computation_placer.cc:177] computation placer already registered. Please check linka

Running CFL K=2...

-- K-Means Clustering Results --
  - Cluster 0 assigned 132 engines.
  - Cluster 1 assigned 117 engines.
---------------------------------
✅ Created sequences: X shape = (11380, 30, 24), y shape = (11380,)
✅ Created sequences: X shape = (3128, 30, 24), y shape = (3128,)
✅ Created sequences: X shape = (10903, 30, 24), y shape = (10903,)
✅ Created sequences: X shape = (3603, 30, 24), y shape = (3603,)
✅ Created sequences: X shape = (10533, 30, 24), y shape = (10533,)
✅ Created sequences: X shape = (2373, 30, 24), y shape = (2373,)
✅ Created sequences: X shape = (9746, 30, 24), y shape = (9746,)
✅ Created sequences: X shape = (2362, 30, 24), y shape = (2362,)


I0000 00:00:1783574171.864202     114 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1783574171.870375     114 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1783574178.827585     165 cuda_dnn.cc:529] Loaded cuDNN version 91002


  [Round 1] Global MAE: 39.8604
  [Round 2] Global MAE: 21.463
  [Round 3] Global MAE: 19.5308
  [Round 4] Global MAE: 19.3772
  [Round 5] CFL one-shot clustering: {0: 0, 1: 0, 2: 0, 3: 1}
  [Round 5] Cluster MAEs: [19.6, 19.6] | Avg: 19.5978
  [Round 6] Cluster MAEs: [19.9, 20.41] | Avg: 20.1524
  [Round 7] Cluster MAEs: [21.67, 20.61] | Avg: 21.1371
  [Round 8] Cluster MAEs: [19.9, 19.99] | Avg: 19.9406
  [Round 9] Cluster MAEs: [20.37, 20.46] | Avg: 20.4131
  [Round 10] Cluster MAEs: [20.6, 21.75] | Avg: 21.1742
  [Round 11] Cluster MAEs: [21.1, 23.64] | Avg: 22.369
  [Round 12] Cluster MAEs: [21.25, 23.01] | Avg: 22.1264
  [Round 13] Cluster MAEs: [21.82, 21.85] | Avg: 21.8329
  [Round 14] Cluster MAEs: [22.24, 23.97] | Avg: 23.1033
  [Round 15] Cluster MAEs: [22.52, 23.37] | Avg: 22.9429
  [Round 16] Cluster MAEs: [22.2, 23.2] | Avg: 22.7004
  [Round 17] Cluster MAEs: [22.11, 23.5] | Avg: 22.8053
  [Round 18] Cluster MAEs: [22.3, 22.65] | Avg: 22.4784
  [Round 19] Cluster MAEs: [2

In [7]:
from run_experiment import run_simulation, run_cfl
print("Running CFL K=2...")
cfl_result = run_cfl('FD004', 2026)
print("CFL:", cfl_result)

Running CFL K=2...

-- K-Means Clustering Results --
  - Cluster 0 assigned 132 engines.
  - Cluster 1 assigned 117 engines.
---------------------------------
✅ Created sequences: X shape = (11559, 30, 24), y shape = (11559,)
✅ Created sequences: X shape = (2949, 30, 24), y shape = (2949,)
✅ Created sequences: X shape = (11576, 30, 24), y shape = (11576,)
✅ Created sequences: X shape = (2930, 30, 24), y shape = (2930,)
✅ Created sequences: X shape = (10377, 30, 24), y shape = (10377,)
✅ Created sequences: X shape = (2529, 30, 24), y shape = (2529,)
✅ Created sequences: X shape = (9810, 30, 24), y shape = (9810,)
✅ Created sequences: X shape = (2298, 30, 24), y shape = (2298,)
  [Round 1] Global MAE: 39.6341
  [Round 2] Global MAE: 20.4021
  [Round 3] Global MAE: 19.334
  [Round 4] Global MAE: 19.3306
  [Round 5] CFL one-shot clustering: {0: 1, 1: 0, 2: 0, 3: 0}
  [Round 5] Cluster MAEs: [19.65, 19.65] | Avg: 19.6459
  [Round 6] Cluster MAEs: [21.69, 21.86] | Avg: 21.7722
  [Round 7] Cl

In [8]:
from run_experiment import run_simulation
print("Running Ditto (cluster-routed)...")
ditto_result = run_simulation('ditto', 'FD004', 42)
print("Ditto:", ditto_result)

Running Ditto (cluster-routed)...

-- K-Means Clustering Results --
  - Cluster 0 assigned 132 engines.
  - Cluster 1 assigned 117 engines.
---------------------------------
✅ Created sequences: X shape = (12060, 30, 24), y shape = (12060,)
✅ Created sequences: X shape = (2448, 30, 24), y shape = (2448,)
✅ Created sequences: X shape = (11660, 30, 24), y shape = (11660,)
✅ Created sequences: X shape = (2846, 30, 24), y shape = (2846,)
✅ Created sequences: X shape = (10505, 30, 24), y shape = (10505,)
✅ Created sequences: X shape = (2401, 30, 24), y shape = (2401,)
✅ Created sequences: X shape = (9866, 30, 24), y shape = (9866,)
✅ Created sequences: X shape = (2242, 30, 24), y shape = (2242,)


INFO flwr 2026-07-09 08:56:19,637 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-09 08:56:28,268	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-09 08:56:31,683 | app.py:210 | Flower VCE: Ray initialized with resources: {'GPU': 2.0, 'accelerator_type:T4': 1.0, 'memory': 15608575181.0, 'node:__internal_head__': 1.0, 'object_store_memory': 6689389363.0, 'CPU': 4.0, 'node:172.19.2.2': 1.0}
INFO flwr 2026-07-09 08:56:31,684 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-07-09 08:56:31,762 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-07-09 08:56:31,764 | server.py:89 | Initializing global parameters
INFO flwr 2026-07-09 08:56:31,764 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-07-09 08:56:31,765 | server.py:91 | Evaluating initial parameters
INFO flwr 2026-07-09 08:

Ditto: {'method': 'ditto', 'dataset': 'FD004', 'seed': 42, 'test_mae': 23.2696, 'nasa_score': 45122.9, 'comm_kb': 28900.78}


In [9]:
from run_experiment import run_simulation
print("Running Ditto (cluster-routed)...")
ditto_result = run_simulation('ditto', 'FD004', 101)
print("Ditto:", ditto_result)

Running Ditto (cluster-routed)...

-- K-Means Clustering Results --
  - Cluster 0 assigned 132 engines.
  - Cluster 1 assigned 117 engines.
---------------------------------
✅ Created sequences: X shape = (11353, 30, 24), y shape = (11353,)
✅ Created sequences: X shape = (3155, 30, 24), y shape = (3155,)
✅ Created sequences: X shape = (11101, 30, 24), y shape = (11101,)
✅ Created sequences: X shape = (3405, 30, 24), y shape = (3405,)
✅ Created sequences: X shape = (10244, 30, 24), y shape = (10244,)
✅ Created sequences: X shape = (2662, 30, 24), y shape = (2662,)
✅ Created sequences: X shape = (9748, 30, 24), y shape = (9748,)
✅ Created sequences: X shape = (2360, 30, 24), y shape = (2360,)


INFO flwr 2026-07-09 10:30:05,503 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-09 10:30:13,759	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-09 10:30:17,392 | app.py:210 | Flower VCE: Ray initialized with resources: {'node:__internal_head__': 1.0, 'node:172.19.2.2': 1.0, 'GPU': 2.0, 'accelerator_type:T4': 1.0, 'memory': 15621537792.0, 'object_store_memory': 6694944768.0, 'CPU': 4.0}
INFO flwr 2026-07-09 10:30:17,392 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-07-09 10:30:17,418 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-07-09 10:30:17,419 | server.py:89 | Initializing global parameters
INFO flwr 2026-07-09 10:30:17,420 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-07-09 10:30:17,421 | server.py:91 | Evaluating initial parameters
INFO flwr 2026-07-09 10:

Ditto: {'method': 'ditto', 'dataset': 'FD004', 'seed': 101, 'test_mae': 23.8484, 'nasa_score': 38520.75, 'comm_kb': 28900.78}


In [10]:
from run_experiment import run_simulation
print("Running Ditto (cluster-routed)...")
ditto_result = run_simulation('ditto', 'FD004', 202)
print("Ditto:", ditto_result)

Running Ditto (cluster-routed)...

-- K-Means Clustering Results --
  - Cluster 0 assigned 132 engines.
  - Cluster 1 assigned 117 engines.
---------------------------------
✅ Created sequences: X shape = (11720, 30, 24), y shape = (11720,)
✅ Created sequences: X shape = (2788, 30, 24), y shape = (2788,)
✅ Created sequences: X shape = (11430, 30, 24), y shape = (11430,)
✅ Created sequences: X shape = (3076, 30, 24), y shape = (3076,)
✅ Created sequences: X shape = (10465, 30, 24), y shape = (10465,)
✅ Created sequences: X shape = (2441, 30, 24), y shape = (2441,)
✅ Created sequences: X shape = (9809, 30, 24), y shape = (9809,)
✅ Created sequences: X shape = (2299, 30, 24), y shape = (2299,)


INFO flwr 2026-07-09 12:07:20,380 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-09 12:07:29,787	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-09 12:07:33,261 | app.py:210 | Flower VCE: Ray initialized with resources: {'CPU': 4.0, 'accelerator_type:T4': 1.0, 'object_store_memory': 6707002982.0, 'GPU': 2.0, 'node:172.19.2.2': 1.0, 'node:__internal_head__': 1.0, 'memory': 15649673626.0}
INFO flwr 2026-07-09 12:07:33,262 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-07-09 12:07:33,282 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-07-09 12:07:33,284 | server.py:89 | Initializing global parameters
INFO flwr 2026-07-09 12:07:33,285 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-07-09 12:07:33,286 | server.py:91 | Evaluating initial parameters
INFO flwr 2026-07-09 12:

Ditto: {'method': 'ditto', 'dataset': 'FD004', 'seed': 202, 'test_mae': 22.9055, 'nasa_score': 27405.26, 'comm_kb': 28900.78}


In [11]:
from run_experiment import run_simulation
print("Running Ditto (cluster-routed)...")
ditto_result = run_simulation('ditto', 'FD004', 303)
print("Ditto:", ditto_result)

Running Ditto (cluster-routed)...

-- K-Means Clustering Results --
  - Cluster 0 assigned 132 engines.
  - Cluster 1 assigned 117 engines.
---------------------------------
✅ Created sequences: X shape = (11380, 30, 24), y shape = (11380,)
✅ Created sequences: X shape = (3128, 30, 24), y shape = (3128,)
✅ Created sequences: X shape = (10903, 30, 24), y shape = (10903,)
✅ Created sequences: X shape = (3603, 30, 24), y shape = (3603,)
✅ Created sequences: X shape = (10533, 30, 24), y shape = (10533,)
✅ Created sequences: X shape = (2373, 30, 24), y shape = (2373,)
✅ Created sequences: X shape = (9746, 30, 24), y shape = (9746,)
✅ Created sequences: X shape = (2362, 30, 24), y shape = (2362,)


INFO flwr 2026-07-09 13:43:58,919 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-09 13:44:08,506	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-09 13:44:12,030 | app.py:210 | Flower VCE: Ray initialized with resources: {'node:__internal_head__': 1.0, 'memory': 15648667239.0, 'node:172.19.2.2': 1.0, 'CPU': 4.0, 'object_store_memory': 6706571673.0, 'accelerator_type:T4': 1.0, 'GPU': 2.0}
INFO flwr 2026-07-09 13:44:12,032 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-07-09 13:44:12,053 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-07-09 13:44:12,054 | server.py:89 | Initializing global parameters
INFO flwr 2026-07-09 13:44:12,055 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-07-09 13:44:12,056 | server.py:91 | Evaluating initial parameters
INFO flwr 2026-07-09 13:

Ditto: {'method': 'ditto', 'dataset': 'FD004', 'seed': 303, 'test_mae': 23.1979, 'nasa_score': 25200.19, 'comm_kb': 28900.78}


In [12]:
from run_experiment import run_simulation
print("Running Ditto (cluster-routed)...")
ditto_result = run_simulation('ditto', 'FD004', 2026)
print("Ditto:", ditto_result)

Running Ditto (cluster-routed)...

-- K-Means Clustering Results --
  - Cluster 0 assigned 132 engines.
  - Cluster 1 assigned 117 engines.
---------------------------------
✅ Created sequences: X shape = (11559, 30, 24), y shape = (11559,)
✅ Created sequences: X shape = (2949, 30, 24), y shape = (2949,)
✅ Created sequences: X shape = (11576, 30, 24), y shape = (11576,)
✅ Created sequences: X shape = (2930, 30, 24), y shape = (2930,)
✅ Created sequences: X shape = (10377, 30, 24), y shape = (10377,)
✅ Created sequences: X shape = (2529, 30, 24), y shape = (2529,)
✅ Created sequences: X shape = (9810, 30, 24), y shape = (9810,)
✅ Created sequences: X shape = (2298, 30, 24), y shape = (2298,)


INFO flwr 2026-07-09 15:23:36,961 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-09 15:23:45,405	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-09 15:23:49,124 | app.py:210 | Flower VCE: Ray initialized with resources: {'accelerator_type:T4': 1.0, 'memory': 15648151143.0, 'node:__internal_head__': 1.0, 'node:172.19.2.2': 1.0, 'GPU': 2.0, 'CPU': 4.0, 'object_store_memory': 6706350489.0}
INFO flwr 2026-07-09 15:23:49,125 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-07-09 15:23:49,145 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-07-09 15:23:49,146 | server.py:89 | Initializing global parameters
INFO flwr 2026-07-09 15:23:49,147 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-07-09 15:23:49,148 | server.py:91 | Evaluating initial parameters
INFO flwr 2026-07-09 15:

Ditto: {'method': 'ditto', 'dataset': 'FD004', 'seed': 2026, 'test_mae': 22.4209, 'nasa_score': 21909.05, 'comm_kb': 28900.78}


In [6]:
from run_experiment import run_simulation
print("Running FedProx")
ditto_result = run_simulation('fedprox', 'FD004', 42)
print("FedProx:", fedprox_result)

E0000 00:00:1783618374.386732     117 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1783618374.448930     117 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1783618374.967239     117 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783618374.967281     117 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783618374.967284     117 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783618374.967287     117 computation_placer.cc:177] computation placer already registered. Please check linka

Running FedProx

-- K-Means Clustering Results --
  - Cluster 0 assigned 132 engines.
  - Cluster 1 assigned 117 engines.
---------------------------------
✅ Created sequences: X shape = (12060, 30, 24), y shape = (12060,)
✅ Created sequences: X shape = (2448, 30, 24), y shape = (2448,)
✅ Created sequences: X shape = (11660, 30, 24), y shape = (11660,)
✅ Created sequences: X shape = (2846, 30, 24), y shape = (2846,)
✅ Created sequences: X shape = (10505, 30, 24), y shape = (10505,)
✅ Created sequences: X shape = (2401, 30, 24), y shape = (2401,)
✅ Created sequences: X shape = (9866, 30, 24), y shape = (9866,)
✅ Created sequences: X shape = (2242, 30, 24), y shape = (2242,)


I0000 00:00:1783618438.388702     117 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1783618438.394537     117 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
INFO flwr 2026-07-09 17:34:00,297 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-09 17:34:08,461	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-09 17:34:11,900 | app.py:210 | Flower VCE: Ray initialized with resources: {'accelerator_type:T4': 1.0, 'CPU': 4.0, 'node:172.19.2.2': 1.0, 'object_store_memory': 9224901427.0, 'memory': 21524769997.0, 'node:__internal_head__': 1.0, 'GPU': 2.0}
INFO flwr 2026-07-09 17:34:11,901 | app.py:224 | Flower VCE: Resources for each Virtu

  [Round 0] Test MAE: 78.7935 | NASA: 1667418.29


(DefaultActor pid=363) I0000 00:00:1783618462.682579     363 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13596 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
(pid=364) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR [repeated 3x across cluster] (Ray deduplicates logs by default. Set RAY_DEDUP_LOGS=0 to disable log deduplication, or see https://docs.ray.io/en/master/ray-observability/user-guides/configure-logging.html#log-deduplication for more options.)
(pid=364) E0000 00:00:1783618453.216798     364 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered [repeated 3x across cluster]
(pid=364) E0000 00:00:1783618453.231909     364 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered [repeated 3x ac

...

  [Round 49] Test MAE: 22.1699 | NASA: 83911.11


DEBUG flwr 2026-07-09 18:23:03,100 | server.py:187 | evaluate_round 49 received 4 results and 0 failures
DEBUG flwr 2026-07-09 18:23:03,101 | server.py:222 | fit_round 50: strategy sampled 4 clients (out of 4)
DEBUG flwr 2026-07-09 18:23:46,701 | server.py:236 | fit_round 50 received 4 results and 0 failures
INFO flwr 2026-07-09 18:23:48,080 | server.py:125 | fit progress: (50, 0.0, {'mae': 22.574443724847608, 'nasa_score': 86212.3472803817}, 2972.4161495959997)
DEBUG flwr 2026-07-09 18:23:48,082 | server.py:173 | evaluate_round 50: strategy sampled 4 clients (out of 4)


  [Round 50] Test MAE: 22.5744 | NASA: 86212.35


DEBUG flwr 2026-07-09 18:23:50,550 | server.py:187 | evaluate_round 50 received 4 results and 0 failures
INFO flwr 2026-07-09 18:23:50,550 | server.py:153 | FL finished in 2974.886299973
INFO flwr 2026-07-09 18:23:50,551 | app.py:225 | app_fit: losses_distributed [(1, 2005.9630908242434), (2, 622.8361835754216), (3, 641.2505258900113), (4, 540.6861129320039), (5, 554.6562505128746), (6, 549.7937843067084), (7, 560.0672762021339), (8, 581.5304760475616), (9, 604.7439208566705), (10, 618.7261957669746), (11, 649.8580686007393), (12, 631.2290869735191), (13, 649.124832680015), (14, 668.1026742955719), (15, 630.8072313030589), (16, 659.9566047286872), (17, 650.6065515420489), (18, 648.9202508702776), (19, 666.5280211376595), (20, 674.4742189883178), (21, 665.1063627796033), (22, 699.725879492217), (23, 686.6005920367161), (24, 677.1830743031686), (25, 688.3815020591642), (26, 729.044103253328), (27, 683.407265147873), (28, 708.1321128038281), (29, 702.8794391175637), (30, 703.9590488252259

NameError: name 'fedprox_result' is not defined

In [6]:
from run_experiment import run_simulation
print("Running FedProx")
fedprox_result = run_simulation('fedprox', 'FD004', 101)
print("FedProx:", fedprox_result)

E0000 00:00:1783623370.242067     115 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1783623370.287193     115 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1783623370.664865     115 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783623370.664921     115 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783623370.664938     115 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783623370.664942     115 computation_placer.cc:177] computation placer already registered. Please check linka

Running FedProx

-- K-Means Clustering Results --
  - Cluster 0 assigned 132 engines.
  - Cluster 1 assigned 117 engines.
---------------------------------
✅ Created sequences: X shape = (11353, 30, 24), y shape = (11353,)
✅ Created sequences: X shape = (3155, 30, 24), y shape = (3155,)
✅ Created sequences: X shape = (11101, 30, 24), y shape = (11101,)
✅ Created sequences: X shape = (3405, 30, 24), y shape = (3405,)
✅ Created sequences: X shape = (10244, 30, 24), y shape = (10244,)
✅ Created sequences: X shape = (2662, 30, 24), y shape = (2662,)
✅ Created sequences: X shape = (9748, 30, 24), y shape = (9748,)
✅ Created sequences: X shape = (2360, 30, 24), y shape = (2360,)


I0000 00:00:1783623431.034438     115 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1783623431.040397     115 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
INFO flwr 2026-07-09 18:57:12,762 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-09 18:57:20,784	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-09 18:57:24,306 | app.py:210 | Flower VCE: Ray initialized with resources: {'node:__internal_head__': 1.0, 'memory': 21527201383.0, 'GPU': 2.0, 'CPU': 4.0, 'object_store_memory': 9225943449.0, 'node:172.19.2.2': 1.0, 'accelerator_type:T4': 1.0}
INFO flwr 2026-07-09 18:57:24,307 | app.py:224 | Flower VCE: Resources for each Virtu

  [Round 0] Test MAE: 79.4685 | NASA: 1757656.36


(DefaultActor pid=364) I0000 00:00:1783623455.301559     364 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13596 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
(pid=365) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR [repeated 3x across cluster] (Ray deduplicates logs by default. Set RAY_DEDUP_LOGS=0 to disable log deduplication, or see https://docs.ray.io/en/master/ray-observability/user-guides/configure-logging.html#log-deduplication for more options.)
(pid=365) E0000 00:00:1783623445.572098     365 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered [repeated 3x across cluster]
(pid=365) E0000 00:00:1783623445.588346     365 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered [repeated 3x ac

...

  [Round 49] Test MAE: 21.7036 | NASA: 32559.66


DEBUG flwr 2026-07-09 19:48:17,279 | server.py:187 | evaluate_round 49 received 4 results and 0 failures
DEBUG flwr 2026-07-09 19:48:17,280 | server.py:222 | fit_round 50: strategy sampled 4 clients (out of 4)
DEBUG flwr 2026-07-09 19:49:08,785 | server.py:236 | fit_round 50 received 4 results and 0 failures
INFO flwr 2026-07-09 19:49:10,127 | server.py:125 | fit progress: (50, 0.0, {'mae': 22.939454493984098, 'nasa_score': 47992.80927792285}, 3102.230487157)
DEBUG flwr 2026-07-09 19:49:10,128 | server.py:173 | evaluate_round 50: strategy sampled 4 clients (out of 4)


  [Round 50] Test MAE: 22.9395 | NASA: 47992.81


DEBUG flwr 2026-07-09 19:49:14,152 | server.py:187 | evaluate_round 50 received 4 results and 0 failures
INFO flwr 2026-07-09 19:49:14,153 | server.py:153 | FL finished in 3106.2557726229998
INFO flwr 2026-07-09 19:49:14,154 | app.py:225 | app_fit: losses_distributed [(1, 1751.0338625175887), (2, 642.9843308335655), (3, 638.4828492422998), (4, 611.3044015749031), (5, 534.1926386099861), (6, 501.00212426268774), (7, 466.93536965592995), (8, 478.1085671826855), (9, 507.00682812761386), (10, 504.49932630509625), (11, 479.7405265843677), (12, 528.2371307462633), (13, 511.21990875628836), (14, 524.6116558740925), (15, 558.3059407759363), (16, 533.1619943425437), (17, 536.0309691919067), (18, 628.2816413269412), (19, 580.8985258434087), (20, 542.7936921710076), (21, 574.0742582210171), (22, 580.6190361559154), (23, 582.3621079753335), (24, 591.5205333711693), (25, 581.3850756226745), (26, 572.721876046588), (27, 592.7006974902883), (28, 581.0178511653658), (29, 597.1606629282221), (30, 588.7

FedProx: {'method': 'fedprox', 'dataset': 'FD004', 'seed': 101, 'test_mae': 22.9395, 'nasa_score': 47992.81, 'comm_kb': 28900.78}


In [7]:
from run_experiment import run_simulation
print("Running FedProx")
fedprox_result = run_simulation('fedprox', 'FD004', 202)
print("FedProx:", fedprox_result)

Running FedProx

-- K-Means Clustering Results --
  - Cluster 0 assigned 132 engines.
  - Cluster 1 assigned 117 engines.
---------------------------------
✅ Created sequences: X shape = (11720, 30, 24), y shape = (11720,)
✅ Created sequences: X shape = (2788, 30, 24), y shape = (2788,)
✅ Created sequences: X shape = (11430, 30, 24), y shape = (11430,)
✅ Created sequences: X shape = (3076, 30, 24), y shape = (3076,)
✅ Created sequences: X shape = (10465, 30, 24), y shape = (10465,)
✅ Created sequences: X shape = (2441, 30, 24), y shape = (2441,)
✅ Created sequences: X shape = (9809, 30, 24), y shape = (9809,)
✅ Created sequences: X shape = (2299, 30, 24), y shape = (2299,)


INFO flwr 2026-07-09 19:50:05,117 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-09 19:50:13,873	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-09 19:50:17,295 | app.py:210 | Flower VCE: Ray initialized with resources: {'accelerator_type:T4': 1.0, 'object_store_memory': 9167749939.0, 'node:__internal_head__': 1.0, 'memory': 21391416525.0, 'node:172.19.2.2': 1.0, 'GPU': 2.0, 'CPU': 4.0}
INFO flwr 2026-07-09 19:50:17,296 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-07-09 19:50:17,316 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-07-09 19:50:17,317 | server.py:89 | Initializing global parameters
INFO flwr 2026-07-09 19:50:17,317 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-07-09 19:50:17,318 | server.py:91 | Evaluating initial parameters
(pid=53115) WARNING: All

  [Round 0] Test MAE: 79.9976 | NASA: 1828224.44


(DefaultActor pid=53115) I0000 00:00:1783626627.792550   53115 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13596 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
(pid=53117) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR [repeated 3x across cluster]
(pid=53117) E0000 00:00:1783626618.436539   53117 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered [repeated 3x across cluster]
(pid=53117) E0000 00:00:1783626618.465767   53117 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered [repeated 3x across cluster]
(pid=53117) W0000 00:00:1783626618.520758   53117 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once. [rep

  [Round 1] Test MAE: 40.1397 | NASA: 22613.60


...

  [Round 49] Test MAE: 20.7974 | NASA: 21003.56


DEBUG flwr 2026-07-09 20:42:36,021 | server.py:187 | evaluate_round 49 received 4 results and 0 failures
DEBUG flwr 2026-07-09 20:42:36,022 | server.py:222 | fit_round 50: strategy sampled 4 clients (out of 4)
DEBUG flwr 2026-07-09 20:43:21,349 | server.py:236 | fit_round 50 received 4 results and 0 failures
INFO flwr 2026-07-09 20:43:22,693 | server.py:125 | fit progress: (50, 0.0, {'mae': 21.666047511562223, 'nasa_score': 23256.657402333723}, 3183.2617391969998)
DEBUG flwr 2026-07-09 20:43:22,694 | server.py:173 | evaluate_round 50: strategy sampled 4 clients (out of 4)


  [Round 50] Test MAE: 21.6660 | NASA: 23256.66


DEBUG flwr 2026-07-09 20:43:25,066 | server.py:187 | evaluate_round 50 received 4 results and 0 failures
INFO flwr 2026-07-09 20:43:25,067 | server.py:153 | FL finished in 3185.635072944
INFO flwr 2026-07-09 20:43:25,068 | app.py:225 | app_fit: losses_distributed [(1, 2152.7753116776025), (2, 827.5023335989625), (3, 698.9771689260649), (4, 653.2653351463223), (5, 612.9775672316866), (6, 539.2106346812441), (7, 577.3153261999867), (8, 605.0528426024654), (9, 551.6290638253447), (10, 584.2768376658701), (11, 551.0274223146328), (12, 534.7547236399038), (13, 617.6311611145319), (14, 621.2469247582723), (15, 608.9496959051695), (16, 652.0601944263276), (17, 615.1605210565073), (18, 619.3102070374202), (19, 638.4176427149764), (20, 618.1581761069857), (21, 628.5407780748366), (22, 622.3378416541296), (23, 638.4037700237216), (24, 692.0862486618773), (25, 647.2617661552761), (26, 657.8565930654669), (27, 660.7749116394665), (28, 674.3318246663719), (29, 678.1793489517332), (30, 666.753116281

FedProx: {'method': 'fedprox', 'dataset': 'FD004', 'seed': 202, 'test_mae': 21.666, 'nasa_score': 23256.66, 'comm_kb': 28900.78}


In [8]:
from run_experiment import run_simulation
print("Running FedProx")
fedprox_result = run_simulation('fedprox', 'FD004', 303)
print("FedProx:", fedprox_result)

Running FedProx

-- K-Means Clustering Results --
  - Cluster 0 assigned 132 engines.
  - Cluster 1 assigned 117 engines.
---------------------------------
✅ Created sequences: X shape = (11380, 30, 24), y shape = (11380,)
✅ Created sequences: X shape = (3128, 30, 24), y shape = (3128,)
✅ Created sequences: X shape = (10903, 30, 24), y shape = (10903,)
✅ Created sequences: X shape = (3603, 30, 24), y shape = (3603,)
✅ Created sequences: X shape = (10533, 30, 24), y shape = (10533,)
✅ Created sequences: X shape = (2373, 30, 24), y shape = (2373,)
✅ Created sequences: X shape = (9746, 30, 24), y shape = (9746,)
✅ Created sequences: X shape = (2362, 30, 24), y shape = (2362,)


INFO flwr 2026-07-09 20:44:10,365 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-09 20:44:22,264	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-09 20:44:25,851 | app.py:210 | Flower VCE: Ray initialized with resources: {'CPU': 4.0, 'node:172.19.2.2': 1.0, 'memory': 14920541799.0, 'GPU': 2.0, 'accelerator_type:T4': 1.0, 'node:__internal_head__': 1.0, 'object_store_memory': 6394517913.0}
INFO flwr 2026-07-09 20:44:25,853 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-07-09 20:44:25,882 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-07-09 20:44:25,884 | server.py:89 | Initializing global parameters
INFO flwr 2026-07-09 20:44:25,885 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-07-09 20:44:25,886 | server.py:91 | Evaluating initial parameters
(pid=107636) WARNING: Al

  [Round 0] Test MAE: 78.9105 | NASA: 1682217.28


(DefaultActor pid=107637) I0000 00:00:1783629878.242926  107637 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13606 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
(pid=107635) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR [repeated 3x across cluster]
(pid=107635) E0000 00:00:1783629868.398712  107635 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered [repeated 3x across cluster]
(pid=107635) E0000 00:00:1783629868.435709  107635 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered [repeated 3x across cluster]
(pid=107635) W0000 00:00:1783629868.621193  107635 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.

  [Round 1] Test MAE: 39.7164 | NASA: 286590.21


...

  [Round 49] Test MAE: 22.9873 | NASA: 131731.38


DEBUG flwr 2026-07-09 21:40:04,939 | server.py:187 | evaluate_round 49 received 4 results and 0 failures
DEBUG flwr 2026-07-09 21:40:04,940 | server.py:222 | fit_round 50: strategy sampled 4 clients (out of 4)
DEBUG flwr 2026-07-09 21:41:30,515 | server.py:236 | fit_round 50 received 4 results and 0 failures
INFO flwr 2026-07-09 21:41:31,986 | server.py:125 | fit progress: (50, 0.0, {'mae': 22.910509863207416, 'nasa_score': 126335.6259190471}, 3424.0637478359995)
DEBUG flwr 2026-07-09 21:41:31,987 | server.py:173 | evaluate_round 50: strategy sampled 4 clients (out of 4)


  [Round 50] Test MAE: 22.9105 | NASA: 126335.63


DEBUG flwr 2026-07-09 21:41:35,938 | server.py:187 | evaluate_round 50 received 4 results and 0 failures
INFO flwr 2026-07-09 21:41:35,939 | server.py:153 | FL finished in 3428.016394129999
INFO flwr 2026-07-09 21:41:35,940 | app.py:225 | app_fit: losses_distributed [(1, 1853.1980680994352), (2, 587.6725169816775), (3, 661.878252328513), (4, 571.1596157114564), (5, 736.2888088469186), (6, 740.657236846812), (7, 739.179621743216), (8, 751.0921339204376), (9, 813.8462228926402), (10, 699.6300748608488), (11, 781.3722311193233), (12, 754.3216321497048), (13, 793.1783622609952), (14, 793.8287513635763), (15, 790.3398153137707), (16, 738.4304832193669), (17, 763.9513795893582), (18, 818.999125482149), (19, 834.0600539200306), (20, 818.4907286234932), (21, 915.5355915021006), (22, 785.9355581760324), (23, 852.3688954357159), (24, 767.7585541468816), (25, 879.2331846015275), (26, 755.1525521191048), (27, 737.9761251718736), (28, 776.2590698582869), (29, 778.2274153047512), (30, 786.6088804214

FedProx: {'method': 'fedprox', 'dataset': 'FD004', 'seed': 303, 'test_mae': 22.9105, 'nasa_score': 126335.63, 'comm_kb': 28900.78}


In [9]:
from run_experiment import run_simulation
print("Running FedProx")
fedprox_result = run_simulation('fedprox', 'FD004', 2026)
print("FedProx:", fedprox_result)

Running FedProx

-- K-Means Clustering Results --
  - Cluster 0 assigned 132 engines.
  - Cluster 1 assigned 117 engines.
---------------------------------
✅ Created sequences: X shape = (11559, 30, 24), y shape = (11559,)
✅ Created sequences: X shape = (2949, 30, 24), y shape = (2949,)
✅ Created sequences: X shape = (11576, 30, 24), y shape = (11576,)
✅ Created sequences: X shape = (2930, 30, 24), y shape = (2930,)
✅ Created sequences: X shape = (10377, 30, 24), y shape = (10377,)
✅ Created sequences: X shape = (2529, 30, 24), y shape = (2529,)
✅ Created sequences: X shape = (9810, 30, 24), y shape = (9810,)
✅ Created sequences: X shape = (2298, 30, 24), y shape = (2298,)


INFO flwr 2026-07-09 21:42:24,516 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-09 21:42:37,312	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-09 21:42:40,884 | app.py:210 | Flower VCE: Ray initialized with resources: {'node:172.19.2.2': 1.0, 'memory': 14968165991.0, 'GPU': 2.0, 'accelerator_type:T4': 1.0, 'node:__internal_head__': 1.0, 'CPU': 4.0, 'object_store_memory': 6414928281.0}
INFO flwr 2026-07-09 21:42:40,885 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-07-09 21:42:40,923 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-07-09 21:42:40,924 | server.py:89 | Initializing global parameters
INFO flwr 2026-07-09 21:42:40,929 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-07-09 21:42:40,930 | server.py:91 | Evaluating initial parameters
(pid=163562) WARNING: Al

  [Round 0] Test MAE: 78.8697 | NASA: 1677290.49


(DefaultActor pid=163560) I0000 00:00:1783633373.688966  163560 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13606 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
(pid=163561) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR [repeated 3x across cluster]
(pid=163561) E0000 00:00:1783633364.595110  163561 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered [repeated 3x across cluster]
(pid=163561) E0000 00:00:1783633364.624787  163561 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered [repeated 3x across cluster]
(pid=163561) W0000 00:00:1783633364.710575  163561 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.

  [Round 1] Test MAE: 40.7485 | NASA: 23215.80


...

  [Round 49] Test MAE: 21.5836 | NASA: 69583.04


DEBUG flwr 2026-07-09 22:34:13,370 | server.py:187 | evaluate_round 49 received 4 results and 0 failures
DEBUG flwr 2026-07-09 22:34:13,370 | server.py:222 | fit_round 50: strategy sampled 4 clients (out of 4)
DEBUG flwr 2026-07-09 22:34:58,747 | server.py:236 | fit_round 50 received 4 results and 0 failures
INFO flwr 2026-07-09 22:35:00,081 | server.py:125 | fit progress: (50, 0.0, {'mae': 22.177349721231767, 'nasa_score': 84984.47651319983}, 3136.974937195999)
DEBUG flwr 2026-07-09 22:35:00,083 | server.py:173 | evaluate_round 50: strategy sampled 4 clients (out of 4)


  [Round 50] Test MAE: 22.1773 | NASA: 84984.48


DEBUG flwr 2026-07-09 22:35:04,012 | server.py:187 | evaluate_round 50 received 4 results and 0 failures
INFO flwr 2026-07-09 22:35:04,013 | server.py:153 | FL finished in 3140.906781382999
INFO flwr 2026-07-09 22:35:04,014 | app.py:225 | app_fit: losses_distributed [(1, 2228.290747380448), (2, 609.4559113381177), (3, 500.3090265232359), (4, 508.94515226133603), (5, 440.06013315743473), (6, 431.0063504411999), (7, 472.65578424919437), (8, 474.58697949599576), (9, 505.49313729904577), (10, 465.9509111729021), (11, 527.0578380793517), (12, 525.0438159847847), (13, 536.7465342709266), (14, 545.2615462313316), (15, 550.0114634154156), (16, 573.5622428923973), (17, 594.9327777653749), (18, 594.7361407010061), (19, 595.3285294762064), (20, 575.2403532470383), (21, 592.9109663255144), (22, 607.0940083760804), (23, 617.1120226265181), (24, 604.8051583760256), (25, 607.1969146557485), (26, 597.5312249525538), (27, 616.6109309660571), (28, 665.9540780856858), (29, 623.212348451976), (30, 623.483

FedProx: {'method': 'fedprox', 'dataset': 'FD004', 'seed': 2026, 'test_mae': 22.1773, 'nasa_score': 84984.48, 'comm_kb': 28900.78}


In [6]:
from run_experiment import run_simulation
print("Checking FedAvg seed 42...")
result = run_simulation('fedavg', 'FD003', 42)y
print("FedAvg 42:", result)

E0000 00:00:1783757052.365403     113 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1783757052.428970     113 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1783757053.004084     113 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783757053.004125     113 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783757053.004128     113 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783757053.004131     113 computation_placer.cc:177] computation placer already registered. Please check linka

Checking FedAvg seed 42...

-- K-Means Clustering Results --
  - Cluster 0 assigned 44 engines.
  - Cluster 1 assigned 56 engines.
---------------------------------
✅ Created sequences: X shape = (5028, 30, 24), y shape = (5028,)
✅ Created sequences: X shape = (1316, 30, 24), y shape = (1316,)
✅ Created sequences: X shape = (4687, 30, 24), y shape = (4687,)
✅ Created sequences: X shape = (1098, 30, 24), y shape = (1098,)
✅ Created sequences: X shape = (4203, 30, 24), y shape = (4203,)
✅ Created sequences: X shape = (1036, 30, 24), y shape = (1036,)
✅ Created sequences: X shape = (3515, 30, 24), y shape = (3515,)
✅ Created sequences: X shape = (937, 30, 24), y shape = (937,)


I0000 00:00:1783757088.571086     113 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1783757088.577041     113 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
INFO flwr 2026-07-11 08:04:50,461 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-11 08:04:58,496	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-11 08:05:01,930 | app.py:210 | Flower VCE: Ray initialized with resources: {'memory': 21670251725.0, 'CPU': 4.0, 'GPU': 2.0, 'object_store_memory': 9287250739.0, 'node:__internal_head__': 1.0, 'accelerator_type:T4': 1.0, 'node:172.19.2.2': 1.0}
INFO flwr 2026-07-11 08:05:01,931 | app.py:224 | Flower VCE: Resources for each Virtu

  [Round 0] Test MAE: 74.5037 | NASA: 463135.23


(DefaultActor pid=394) I0000 00:00:1783757112.313411     394 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13644 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
(pid=393) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR [repeated 3x across cluster] (Ray deduplicates logs by default. Set RAY_DEDUP_LOGS=0 to disable log deduplication, or see https://docs.ray.io/en/master/ray-observability/user-guides/configure-logging.html#log-deduplication for more options.)
(pid=393) E0000 00:00:1783757103.326318     393 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered [repeated 3x across cluster]
(pid=393) E0000 00:00:1783757103.338777     393 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered [repeated 3x ac

...

  [Round 49] Test MAE: 13.1117 | NASA: 434.32


DEBUG flwr 2026-07-11 08:31:35,372 | server.py:187 | evaluate_round 49 received 4 results and 0 failures
DEBUG flwr 2026-07-11 08:31:35,373 | server.py:222 | fit_round 50: strategy sampled 4 clients (out of 4)
DEBUG flwr 2026-07-11 08:32:08,840 | server.py:236 | fit_round 50 received 4 results and 0 failures
INFO flwr 2026-07-11 08:32:10,192 | server.py:125 | fit progress: (50, 0.0, {'mae': 12.857542839050293, 'nasa_score': 446.45598730068446}, 1622.128459552)
DEBUG flwr 2026-07-11 08:32:10,193 | server.py:173 | evaluate_round 50: strategy sampled 4 clients (out of 4)


  [Round 50] Test MAE: 12.8575 | NASA: 446.46


DEBUG flwr 2026-07-11 08:32:13,408 | server.py:187 | evaluate_round 50 received 4 results and 0 failures
INFO flwr 2026-07-11 08:32:13,408 | server.py:153 | FL finished in 1625.344364501
INFO flwr 2026-07-11 08:32:13,409 | app.py:225 | app_fit: losses_distributed [(1, 1013.3297237816231), (2, 596.113857390154), (3, 565.1376789093887), (4, 554.8536562827228), (5, 586.9344755389682), (6, 579.801625794584), (7, 559.1612601279126), (8, 543.9121797873378), (9, 507.05104662145436), (10, 631.3485088639686), (11, 635.8180142140263), (12, 589.4872741420965), (13, 584.9531574931862), (14, 528.499678031551), (15, 505.7096074358434), (16, 600.9696685464066), (17, 492.1449748713796), (18, 476.3781080695962), (19, 614.3108788884417), (20, 569.3525726895738), (21, 502.6929359966458), (22, 516.3570861468588), (23, 533.6530191157601), (24, 533.4723546219655), (25, 583.3371362627463), (26, 670.6280801397862), (27, 568.8349907246596), (28, 564.9977279951776), (29, 494.9435545288949), (30, 535.23232655887

FedAvg 42: {'method': 'fedavg', 'dataset': 'FD003', 'seed': 42, 'test_mae': 12.8575, 'nasa_score': 446.46, 'comm_kb': 28900.78}


In [6]:
from run_experiment import run_simulation
print("Checking FedPer seed 42...")
result = run_simulation('fedper', 'FD003', 42)
print("FedPer 42:", result)

E0000 00:00:1783843856.138931     113 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1783843856.246583     113 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1783843857.163569     113 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783843857.163607     113 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783843857.163610     113 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783843857.163613     113 computation_placer.cc:177] computation placer already registered. Please check linka

Checking FedPer seed 42...

-- K-Means Clustering Results --
  - Cluster 0 assigned 44 engines.
  - Cluster 1 assigned 56 engines.
---------------------------------
✅ Created sequences: X shape = (5028, 30, 24), y shape = (5028,)
✅ Created sequences: X shape = (1316, 30, 24), y shape = (1316,)
✅ Created sequences: X shape = (4687, 30, 24), y shape = (4687,)
✅ Created sequences: X shape = (1098, 30, 24), y shape = (1098,)
✅ Created sequences: X shape = (4203, 30, 24), y shape = (4203,)
✅ Created sequences: X shape = (1036, 30, 24), y shape = (1036,)
✅ Created sequences: X shape = (3515, 30, 24), y shape = (3515,)
✅ Created sequences: X shape = (937, 30, 24), y shape = (937,)


I0000 00:00:1783843898.171164     113 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1783843898.177367     113 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
INFO flwr 2026-07-12 08:11:40,492 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-12 08:11:48,864	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-12 08:11:52,454 | app.py:210 | Flower VCE: Ray initialized with resources: {'memory': 21672230093.0, 'GPU': 2.0, 'object_store_memory': 9288098611.0, 'node:172.19.2.2': 1.0, 'node:__internal_head__': 1.0, 'accelerator_type:T4': 1.0, 'CPU': 4.0}
INFO flwr 2026-07-12 08:11:52,454 | app.py:224 | Flower VCE: Resources for each Virtu

FedPer 42: {'method': 'fedper', 'dataset': 'FD003', 'seed': 42, 'test_mae': 16.6874, 'nasa_score': 988.39, 'comm_kb': 27650.0}


In [7]:
from run_experiment import run_simulation
print("Checking FedPer seed 202...")
result = run_simulation('fedper', 'FD003', 202)
print("FedPer 202:", result)

Checking FedPer seed 202...

-- K-Means Clustering Results --
  - Cluster 0 assigned 44 engines.
  - Cluster 1 assigned 56 engines.
---------------------------------
✅ Created sequences: X shape = (4485, 30, 24), y shape = (4485,)
✅ Created sequences: X shape = (1859, 30, 24), y shape = (1859,)
✅ Created sequences: X shape = (4383, 30, 24), y shape = (4383,)
✅ Created sequences: X shape = (1402, 30, 24), y shape = (1402,)
✅ Created sequences: X shape = (4176, 30, 24), y shape = (4176,)
✅ Created sequences: X shape = (1063, 30, 24), y shape = (1063,)
✅ Created sequences: X shape = (3551, 30, 24), y shape = (3551,)
✅ Created sequences: X shape = (901, 30, 24), y shape = (901,)


INFO flwr 2026-07-12 08:38:09,512 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-12 08:38:22,114	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-12 08:38:25,570 | app.py:210 | Flower VCE: Ray initialized with resources: {'object_store_memory': 6610324684.0, 'memory': 15424090932.0, 'node:__internal_head__': 1.0, 'CPU': 4.0, 'node:172.19.2.2': 1.0, 'GPU': 2.0, 'accelerator_type:T4': 1.0}
INFO flwr 2026-07-12 08:38:25,571 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-07-12 08:38:25,599 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-07-12 08:38:25,601 | server.py:89 | Initializing global parameters
INFO flwr 2026-07-12 08:38:25,603 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-07-12 08:38:25,606 | server.py:91 | Evaluating initial parameters
INFO flwr 2026-07-12 08:

FedPer 202: {'method': 'fedper', 'dataset': 'FD003', 'seed': 202, 'test_mae': 13.9603, 'nasa_score': 1495.44, 'comm_kb': 27650.0}


In [8]:
from run_experiment import run_simulation
print("Checking FedPer seed 101...")
result = run_simulation('fedper', 'FD003', 101)
print("FedPer 101:", result)

Checking FedPer seed 101...

-- K-Means Clustering Results --
  - Cluster 0 assigned 44 engines.
  - Cluster 1 assigned 56 engines.
---------------------------------
✅ Created sequences: X shape = (5107, 30, 24), y shape = (5107,)
✅ Created sequences: X shape = (1237, 30, 24), y shape = (1237,)
✅ Created sequences: X shape = (4174, 30, 24), y shape = (4174,)
✅ Created sequences: X shape = (1611, 30, 24), y shape = (1611,)
✅ Created sequences: X shape = (4214, 30, 24), y shape = (4214,)
✅ Created sequences: X shape = (1025, 30, 24), y shape = (1025,)
✅ Created sequences: X shape = (3513, 30, 24), y shape = (3513,)
✅ Created sequences: X shape = (939, 30, 24), y shape = (939,)


INFO flwr 2026-07-12 09:04:40,077 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-12 09:04:51,264	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-12 09:04:54,724 | app.py:210 | Flower VCE: Ray initialized with resources: {'CPU': 4.0, 'GPU': 2.0, 'accelerator_type:T4': 1.0, 'node:__internal_head__': 1.0, 'node:172.19.2.2': 1.0, 'memory': 15417381684.0, 'object_store_memory': 6607449292.0}
INFO flwr 2026-07-12 09:04:54,725 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-07-12 09:04:54,755 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-07-12 09:04:54,756 | server.py:89 | Initializing global parameters
INFO flwr 2026-07-12 09:04:54,757 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-07-12 09:04:54,758 | server.py:91 | Evaluating initial parameters
INFO flwr 2026-07-12 09:

FedPer 101: {'method': 'fedper', 'dataset': 'FD003', 'seed': 101, 'test_mae': 13.0888, 'nasa_score': 715.8, 'comm_kb': 27650.0}


In [9]:
from run_experiment import run_simulation
print("Checking FedPer seed 303...")
result = run_simulation('fedper', 'FD003', 303)
print("FedPer 303:", result)

Checking FedPer seed 303...

-- K-Means Clustering Results --
  - Cluster 0 assigned 44 engines.
  - Cluster 1 assigned 56 engines.
---------------------------------
✅ Created sequences: X shape = (4898, 30, 24), y shape = (4898,)
✅ Created sequences: X shape = (1446, 30, 24), y shape = (1446,)
✅ Created sequences: X shape = (4804, 30, 24), y shape = (4804,)
✅ Created sequences: X shape = (981, 30, 24), y shape = (981,)
✅ Created sequences: X shape = (4097, 30, 24), y shape = (4097,)
✅ Created sequences: X shape = (1142, 30, 24), y shape = (1142,)
✅ Created sequences: X shape = (3571, 30, 24), y shape = (3571,)
✅ Created sequences: X shape = (881, 30, 24), y shape = (881,)


INFO flwr 2026-07-12 09:31:02,067 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-12 09:31:12,051	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-12 09:31:15,537 | app.py:210 | Flower VCE: Ray initialized with resources: {'memory': 15415913677.0, 'object_store_memory': 6606820147.0, 'node:172.19.2.2': 1.0, 'GPU': 2.0, 'CPU': 4.0, 'node:__internal_head__': 1.0, 'accelerator_type:T4': 1.0}
INFO flwr 2026-07-12 09:31:15,538 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-07-12 09:31:15,562 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-07-12 09:31:15,563 | server.py:89 | Initializing global parameters
INFO flwr 2026-07-12 09:31:15,565 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-07-12 09:31:15,567 | server.py:91 | Evaluating initial parameters
INFO flwr 2026-07-12 09:

FedPer 303: {'method': 'fedper', 'dataset': 'FD003', 'seed': 303, 'test_mae': 12.9368, 'nasa_score': 764.05, 'comm_kb': 27650.0}


In [10]:
from run_experiment import run_simulation
print("Checking FedPer seed 2026...")
result = run_simulation('fedper', 'FD003', 2026)
print("FedPer 2026:", result)

Checking FedPer seed 2026...

-- K-Means Clustering Results --
  - Cluster 0 assigned 44 engines.
  - Cluster 1 assigned 56 engines.
---------------------------------
✅ Created sequences: X shape = (4737, 30, 24), y shape = (4737,)
✅ Created sequences: X shape = (1607, 30, 24), y shape = (1607,)
✅ Created sequences: X shape = (4658, 30, 24), y shape = (4658,)
✅ Created sequences: X shape = (1127, 30, 24), y shape = (1127,)
✅ Created sequences: X shape = (4213, 30, 24), y shape = (4213,)
✅ Created sequences: X shape = (1026, 30, 24), y shape = (1026,)
✅ Created sequences: X shape = (3544, 30, 24), y shape = (3544,)
✅ Created sequences: X shape = (908, 30, 24), y shape = (908,)


INFO flwr 2026-07-12 09:57:13,810 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-12 09:57:22,271	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-12 09:57:25,644 | app.py:210 | Flower VCE: Ray initialized with resources: {'object_store_memory': 9212678553.0, 'GPU': 2.0, 'memory': 21496249959.0, 'CPU': 4.0, 'node:172.19.2.2': 1.0, 'node:__internal_head__': 1.0, 'accelerator_type:T4': 1.0}
INFO flwr 2026-07-12 09:57:25,645 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-07-12 09:57:25,665 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-07-12 09:57:25,667 | server.py:89 | Initializing global parameters
INFO flwr 2026-07-12 09:57:25,668 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-07-12 09:57:25,669 | server.py:91 | Evaluating initial parameters
INFO flwr 2026-07-12 09:

FedPer 2026: {'method': 'fedper', 'dataset': 'FD003', 'seed': 2026, 'test_mae': 13.4409, 'nasa_score': 571.97, 'comm_kb': 27650.0}


In [11]:
from run_experiment import run_simulation, run_cfl
print("Running CFL K=2...")
cfl_result = run_cfl('FD003', 42)
print("CFL:", cfl_result)

Running CFL K=2...

-- K-Means Clustering Results --
  - Cluster 0 assigned 44 engines.
  - Cluster 1 assigned 56 engines.
---------------------------------
✅ Created sequences: X shape = (5028, 30, 24), y shape = (5028,)
✅ Created sequences: X shape = (1316, 30, 24), y shape = (1316,)
✅ Created sequences: X shape = (4687, 30, 24), y shape = (4687,)
✅ Created sequences: X shape = (1098, 30, 24), y shape = (1098,)
✅ Created sequences: X shape = (4203, 30, 24), y shape = (4203,)
✅ Created sequences: X shape = (1036, 30, 24), y shape = (1036,)
✅ Created sequences: X shape = (3515, 30, 24), y shape = (3515,)
✅ Created sequences: X shape = (937, 30, 24), y shape = (937,)
  [Round 1] Global MAE: 22.8881
  [Round 2] Global MAE: 16.1772
  [Round 3] Global MAE: 14.703
  [Round 4] Global MAE: 14.9558
  [Round 5] CFL one-shot clustering: {0: 0, 1: 0, 2: 1, 3: 1}
  [Round 5] Cluster MAEs: [14.71, 14.71] | Avg: 14.7123
  [Round 6] Cluster MAEs: [21.56, 24.19] | Avg: 22.8757
  [Round 7] Cluster MAEs

In [12]:
from run_experiment import run_simulation, run_cfl
print("Running CFL K=2...")
cfl_result = run_cfl('FD003', 101)
print("CFL:", cfl_result)

Running CFL K=2...

-- K-Means Clustering Results --
  - Cluster 0 assigned 44 engines.
  - Cluster 1 assigned 56 engines.
---------------------------------
✅ Created sequences: X shape = (5107, 30, 24), y shape = (5107,)
✅ Created sequences: X shape = (1237, 30, 24), y shape = (1237,)
✅ Created sequences: X shape = (4174, 30, 24), y shape = (4174,)
✅ Created sequences: X shape = (1611, 30, 24), y shape = (1611,)
✅ Created sequences: X shape = (4214, 30, 24), y shape = (4214,)
✅ Created sequences: X shape = (1025, 30, 24), y shape = (1025,)
✅ Created sequences: X shape = (3513, 30, 24), y shape = (3513,)
✅ Created sequences: X shape = (939, 30, 24), y shape = (939,)
  [Round 1] Global MAE: 25.4202
  [Round 2] Global MAE: 16.6064
  [Round 3] Global MAE: 16.1858
  [Round 4] Global MAE: 14.6863
  [Round 5] CFL one-shot clustering: {0: 0, 1: 0, 2: 1, 3: 1}
  [Round 5] Cluster MAEs: [15.36, 15.36] | Avg: 15.3575
  [Round 6] Cluster MAEs: [24.27, 19.78] | Avg: 22.0272
  [Round 7] Cluster MAE

In [13]:
from run_experiment import run_simulation, run_cfl
print("Running CFL K=2...")
cfl_result = run_cfl('FD003', 202)
print("CFL:", cfl_result)

Running CFL K=2...

-- K-Means Clustering Results --
  - Cluster 0 assigned 44 engines.
  - Cluster 1 assigned 56 engines.
---------------------------------
✅ Created sequences: X shape = (4485, 30, 24), y shape = (4485,)
✅ Created sequences: X shape = (1859, 30, 24), y shape = (1859,)
✅ Created sequences: X shape = (4383, 30, 24), y shape = (4383,)
✅ Created sequences: X shape = (1402, 30, 24), y shape = (1402,)
✅ Created sequences: X shape = (4176, 30, 24), y shape = (4176,)
✅ Created sequences: X shape = (1063, 30, 24), y shape = (1063,)
✅ Created sequences: X shape = (3551, 30, 24), y shape = (3551,)
✅ Created sequences: X shape = (901, 30, 24), y shape = (901,)
  [Round 1] Global MAE: 20.3932
  [Round 2] Global MAE: 16.6922
  [Round 3] Global MAE: 14.6852
  [Round 4] Global MAE: 17.272
  [Round 5] CFL one-shot clustering: {0: 0, 1: 0, 2: 1, 3: 1}
  [Round 5] Cluster MAEs: [13.78, 13.78] | Avg: 13.7825
  [Round 6] Cluster MAEs: [25.93, 24.69] | Avg: 25.3084
  [Round 7] Cluster MAEs

In [14]:
from run_experiment import run_simulation, run_cfl
print("Running CFL K=2...")
cfl_result = run_cfl('FD003', 303)
print("CFL:", cfl_result)

Running CFL K=2...

-- K-Means Clustering Results --
  - Cluster 0 assigned 44 engines.
  - Cluster 1 assigned 56 engines.
---------------------------------
✅ Created sequences: X shape = (4898, 30, 24), y shape = (4898,)
✅ Created sequences: X shape = (1446, 30, 24), y shape = (1446,)
✅ Created sequences: X shape = (4804, 30, 24), y shape = (4804,)
✅ Created sequences: X shape = (981, 30, 24), y shape = (981,)
✅ Created sequences: X shape = (4097, 30, 24), y shape = (4097,)
✅ Created sequences: X shape = (1142, 30, 24), y shape = (1142,)
✅ Created sequences: X shape = (3571, 30, 24), y shape = (3571,)
✅ Created sequences: X shape = (881, 30, 24), y shape = (881,)
  [Round 1] Global MAE: 19.7973
  [Round 2] Global MAE: 16.9176
  [Round 3] Global MAE: 14.6748
  [Round 4] Global MAE: 13.945
  [Round 5] CFL one-shot clustering: {0: 0, 1: 0, 2: 1, 3: 1}
  [Round 5] Cluster MAEs: [16.2, 16.2] | Avg: 16.1998
  [Round 6] Cluster MAEs: [24.66, 16.99] | Avg: 20.8256
  [Round 7] Cluster MAEs: [2

In [15]:
from run_experiment import run_simulation, run_cfl
print("Running CFL K=2...")
cfl_result = run_cfl('FD003', 2026)
print("CFL:", cfl_result)

Running CFL K=2...

-- K-Means Clustering Results --
  - Cluster 0 assigned 44 engines.
  - Cluster 1 assigned 56 engines.
---------------------------------
✅ Created sequences: X shape = (4737, 30, 24), y shape = (4737,)
✅ Created sequences: X shape = (1607, 30, 24), y shape = (1607,)
✅ Created sequences: X shape = (4658, 30, 24), y shape = (4658,)
✅ Created sequences: X shape = (1127, 30, 24), y shape = (1127,)
✅ Created sequences: X shape = (4213, 30, 24), y shape = (4213,)
✅ Created sequences: X shape = (1026, 30, 24), y shape = (1026,)
✅ Created sequences: X shape = (3544, 30, 24), y shape = (3544,)
✅ Created sequences: X shape = (908, 30, 24), y shape = (908,)
  [Round 1] Global MAE: 27.6083
  [Round 2] Global MAE: 19.1505
  [Round 3] Global MAE: 19.0357
  [Round 4] Global MAE: 19.4798
  [Round 5] CFL one-shot clustering: {0: 0, 1: 0, 2: 1, 3: 1}
  [Round 5] Cluster MAEs: [18.71, 18.71] | Avg: 18.7115
  [Round 6] Cluster MAEs: [25.67, 23.08] | Avg: 24.3739
  [Round 7] Cluster MAE

In [6]:
from run_experiment import run_simulation
print("Running FedProx")
fedprox_result = run_simulation('fedprox', 'FD003', 101)
print("FedProx:", fedprox_result)

E0000 00:00:1783927156.100112     112 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1783927156.163261     112 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1783927156.659369     112 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783927156.659418     112 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783927156.659421     112 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783927156.659424     112 computation_placer.cc:177] computation placer already registered. Please check linka

Running FedProx

-- K-Means Clustering Results --
  - Cluster 0 assigned 44 engines.
  - Cluster 1 assigned 56 engines.
---------------------------------
✅ Created sequences: X shape = (5107, 30, 24), y shape = (5107,)
✅ Created sequences: X shape = (1237, 30, 24), y shape = (1237,)
✅ Created sequences: X shape = (4174, 30, 24), y shape = (4174,)
✅ Created sequences: X shape = (1611, 30, 24), y shape = (1611,)
✅ Created sequences: X shape = (4214, 30, 24), y shape = (4214,)
✅ Created sequences: X shape = (1025, 30, 24), y shape = (1025,)
✅ Created sequences: X shape = (3513, 30, 24), y shape = (3513,)
✅ Created sequences: X shape = (939, 30, 24), y shape = (939,)


I0000 00:00:1783927192.178435     112 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1783927192.184524     112 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
INFO flwr 2026-07-13 07:19:54,093 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-13 07:20:02,028	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-13 07:20:05,664 | app.py:210 | Flower VCE: Ray initialized with resources: {'node:172.19.2.2': 1.0, 'accelerator_type:T4': 1.0, 'memory': 21672938292.0, 'node:__internal_head__': 1.0, 'GPU': 2.0, 'CPU': 4.0, 'object_store_memory': 9288402124.0}
INFO flwr 2026-07-13 07:20:05,665 | app.py:224 | Flower VCE: Resources for each Virtu

  [Round 0] Test MAE: 74.0147 | NASA: 448204.36


(DefaultActor pid=362) I0000 00:00:1783927215.948596     362 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13644 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
(pid=360) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR [repeated 3x across cluster] (Ray deduplicates logs by default. Set RAY_DEDUP_LOGS=0 to disable log deduplication, or see https://docs.ray.io/en/master/ray-observability/user-guides/configure-logging.html#log-deduplication for more options.)
(pid=360) E0000 00:00:1783927206.886009     360 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered [repeated 3x across cluster]
(pid=360) E0000 00:00:1783927206.909782     360 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered [repeated 3x ac

...

  [Round 49] Test MAE: 12.8538 | NASA: 683.93


DEBUG flwr 2026-07-13 07:46:49,031 | server.py:187 | evaluate_round 49 received 4 results and 0 failures
DEBUG flwr 2026-07-13 07:46:49,032 | server.py:222 | fit_round 50: strategy sampled 4 clients (out of 4)
DEBUG flwr 2026-07-13 07:47:17,001 | server.py:236 | fit_round 50 received 4 results and 0 failures
INFO flwr 2026-07-13 07:47:18,276 | server.py:125 | fit progress: (50, 0.0, {'mae': 12.397984085083008, 'nasa_score': 505.55092598083695}, 1629.022432513)
DEBUG flwr 2026-07-13 07:47:18,277 | server.py:173 | evaluate_round 50: strategy sampled 4 clients (out of 4)


  [Round 50] Test MAE: 12.3980 | NASA: 505.55


DEBUG flwr 2026-07-13 07:47:20,276 | server.py:187 | evaluate_round 50 received 4 results and 0 failures
INFO flwr 2026-07-13 07:47:20,277 | server.py:153 | FL finished in 1631.023606383
INFO flwr 2026-07-13 07:47:20,278 | app.py:225 | app_fit: losses_distributed [(1, 962.7987584774432), (2, 440.2619508155068), (3, 477.8556038974626), (4, 462.75196911311605), (5, 506.07092888277964), (6, 462.7429174548472), (7, 457.6885130542175), (8, 395.46895918366516), (9, 399.80197919812287), (10, 427.57162047502703), (11, 423.75678632423865), (12, 475.949207594468), (13, 406.4064223621651), (14, 415.5185880399404), (15, 340.9518584253782), (16, 331.37592817958154), (17, 354.08680921698846), (18, 354.75585958428513), (19, 343.77025446967093), (20, 316.04757607984027), (21, 351.8397754596257), (22, 339.4187444241366), (23, 297.1343891812877), (24, 318.5000103057075), (25, 341.25787561532366), (26, 306.9613516943116), (27, 321.49059458008827), (28, 318.2239386700434), (29, 310.0823454313841), (30, 28

FedProx: {'method': 'fedprox', 'dataset': 'FD003', 'seed': 101, 'test_mae': 12.398, 'nasa_score': 505.55, 'comm_kb': 28900.78}


In [7]:
from run_experiment import run_simulation
print("Running FedProx")
fedprox_result = run_simulation('fedprox', 'FD003', 42)
print("FedProx:", fedprox_result)

Running FedProx

-- K-Means Clustering Results --
  - Cluster 0 assigned 44 engines.
  - Cluster 1 assigned 56 engines.
---------------------------------
✅ Created sequences: X shape = (5028, 30, 24), y shape = (5028,)
✅ Created sequences: X shape = (1316, 30, 24), y shape = (1316,)
✅ Created sequences: X shape = (4687, 30, 24), y shape = (4687,)
✅ Created sequences: X shape = (1098, 30, 24), y shape = (1098,)
✅ Created sequences: X shape = (4203, 30, 24), y shape = (4203,)
✅ Created sequences: X shape = (1036, 30, 24), y shape = (1036,)
✅ Created sequences: X shape = (3515, 30, 24), y shape = (3515,)
✅ Created sequences: X shape = (937, 30, 24), y shape = (937,)


INFO flwr 2026-07-13 07:47:40,260 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-13 07:47:50,517	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-13 07:47:54,504 | app.py:210 | Flower VCE: Ray initialized with resources: {'memory': 15145009152.0, 'GPU': 2.0, 'CPU': 4.0, 'object_store_memory': 6490718208.0, 'node:172.19.2.2': 1.0, 'accelerator_type:T4': 1.0, 'node:__internal_head__': 1.0}
INFO flwr 2026-07-13 07:47:54,505 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-07-13 07:47:54,533 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-07-13 07:47:54,534 | server.py:89 | Initializing global parameters
INFO flwr 2026-07-13 07:47:54,535 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-07-13 07:47:54,536 | server.py:91 | Evaluating initial parameters
INFO flwr 2026-07-13 07:

  [Round 0] Test MAE: 73.9934 | NASA: 444080.60


(pid=54178) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
(pid=54178) E0000 00:00:1783928877.027988   54178 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
(pid=54179) E0000 00:00:1783928877.053611   54179 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
(pid=54179) W0000 00:00:1783928877.080093   54179 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
(pid=54179) W0000 00:00:1783928877.080127   54179 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
(pid=54179) W0000 00:00:1783928877.080132   54179 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid lin

  [Round 1] Test MAE: 27.7982 | NASA: 18648.31


...

  [Round 49] Test MAE: 11.0264 | NASA: 673.90


DEBUG flwr 2026-07-13 08:16:14,837 | server.py:187 | evaluate_round 49 received 4 results and 0 failures
DEBUG flwr 2026-07-13 08:16:14,837 | server.py:222 | fit_round 50: strategy sampled 4 clients (out of 4)
DEBUG flwr 2026-07-13 08:16:44,892 | server.py:236 | fit_round 50 received 4 results and 0 failures
INFO flwr 2026-07-13 08:16:46,195 | server.py:125 | fit progress: (50, 0.0, {'mae': 11.447540550231933, 'nasa_score': 583.6318430614991}, 1729.6562125679998)
DEBUG flwr 2026-07-13 08:16:46,196 | server.py:173 | evaluate_round 50: strategy sampled 4 clients (out of 4)


  [Round 50] Test MAE: 11.4475 | NASA: 583.63


DEBUG flwr 2026-07-13 08:16:49,573 | server.py:187 | evaluate_round 50 received 4 results and 0 failures
INFO flwr 2026-07-13 08:16:49,574 | server.py:153 | FL finished in 1733.035824439
INFO flwr 2026-07-13 08:16:49,576 | app.py:225 | app_fit: losses_distributed [(1, 1070.5197265291094), (2, 567.2007406776903), (3, 448.6840754853095), (4, 434.5047929710317), (5, 408.34406196519456), (6, 380.06732929021916), (7, 368.810814862048), (8, 373.84327219503734), (9, 356.6162805498557), (10, 334.31798656942743), (11, 325.26216603738453), (12, 368.1295664995766), (13, 377.19776139129993), (14, 354.9902213179008), (15, 348.05904155911196), (16, 335.5016533195334), (17, 380.46385595296874), (18, 404.6064928314369), (19, 330.5691429539914), (20, 338.52080077290236), (21, 392.23939495725784), (22, 326.20204602587813), (23, 321.96458104283386), (24, 317.37040811664343), (25, 296.06216050846473), (26, 317.008697516722), (27, 324.88135013271636), (28, 278.07762794677234), (29, 277.9871144308869), (30,

FedProx: {'method': 'fedprox', 'dataset': 'FD003', 'seed': 42, 'test_mae': 11.4475, 'nasa_score': 583.63, 'comm_kb': 28900.78}


In [8]:
from run_experiment import run_simulation
print("Running FedProx")
fedprox_result = run_simulation('fedprox', 'FD003', 202)
print("FedProx:", fedprox_result)

Running FedProx

-- K-Means Clustering Results --
  - Cluster 0 assigned 44 engines.
  - Cluster 1 assigned 56 engines.
---------------------------------
✅ Created sequences: X shape = (4485, 30, 24), y shape = (4485,)
✅ Created sequences: X shape = (1859, 30, 24), y shape = (1859,)
✅ Created sequences: X shape = (4383, 30, 24), y shape = (4383,)
✅ Created sequences: X shape = (1402, 30, 24), y shape = (1402,)
✅ Created sequences: X shape = (4176, 30, 24), y shape = (4176,)
✅ Created sequences: X shape = (1063, 30, 24), y shape = (1063,)
✅ Created sequences: X shape = (3551, 30, 24), y shape = (3551,)
✅ Created sequences: X shape = (901, 30, 24), y shape = (901,)


INFO flwr 2026-07-13 08:17:09,908 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-13 08:17:20,149	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-13 08:17:23,857 | app.py:210 | Flower VCE: Ray initialized with resources: {'node:__internal_head__': 1.0, 'GPU': 2.0, 'CPU': 4.0, 'memory': 15140702618.0, 'object_store_memory': 6488872550.0, 'node:172.19.2.2': 1.0, 'accelerator_type:T4': 1.0}
INFO flwr 2026-07-13 08:17:23,858 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-07-13 08:17:23,890 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-07-13 08:17:23,893 | server.py:89 | Initializing global parameters
INFO flwr 2026-07-13 08:17:23,894 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-07-13 08:17:23,896 | server.py:91 | Evaluating initial parameters
(pid=110128) WARNING: Al

  [Round 0] Test MAE: 73.4275 | NASA: 434171.71


(DefaultActor pid=110128) I0000 00:00:1783930657.063362  110128 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13606 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
(pid=110129) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR [repeated 3x across cluster]
(pid=110129) E0000 00:00:1783930646.442631  110129 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered [repeated 3x across cluster]
(pid=110126) E0000 00:00:1783930646.509033  110126 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered [repeated 3x across cluster]
(pid=110129) W0000 00:00:1783930646.519498  110129 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.

  [Round 1] Test MAE: 25.4459 | NASA: 33335.04


...

  [Round 49] Test MAE: 13.4867 | NASA: 691.07


DEBUG flwr 2026-07-13 08:45:03,468 | server.py:187 | evaluate_round 49 received 4 results and 0 failures
DEBUG flwr 2026-07-13 08:45:03,469 | server.py:222 | fit_round 50: strategy sampled 4 clients (out of 4)
DEBUG flwr 2026-07-13 08:45:36,191 | server.py:236 | fit_round 50 received 4 results and 0 failures
INFO flwr 2026-07-13 08:45:37,472 | server.py:125 | fit progress: (50, 0.0, {'mae': 15.662909698486327, 'nasa_score': 2142.7265264520142}, 1691.5010730579997)
DEBUG flwr 2026-07-13 08:45:37,473 | server.py:173 | evaluate_round 50: strategy sampled 4 clients (out of 4)


  [Round 50] Test MAE: 15.6629 | NASA: 2142.73


DEBUG flwr 2026-07-13 08:45:40,738 | server.py:187 | evaluate_round 50 received 4 results and 0 failures
INFO flwr 2026-07-13 08:45:40,739 | server.py:153 | FL finished in 1694.7687517370005
INFO flwr 2026-07-13 08:45:40,740 | app.py:225 | app_fit: losses_distributed [(1, 662.1794288189901), (2, 608.321875648316), (3, 579.6192743065939), (4, 493.0703447113767), (5, 588.7449526203649), (6, 501.67293582021904), (7, 400.7192810233814), (8, 471.10571637167317), (9, 406.4922085615094), (10, 453.5233528801585), (11, 450.01087357370477), (12, 381.1348090388102), (13, 413.77589483379745), (14, 401.49545809659094), (15, 328.90777418510766), (16, 370.5978727933893), (17, 312.1072895017195), (18, 295.4890607974404), (19, 330.9284781219628), (20, 331.6189995899839), (21, 302.8347694214451), (22, 265.5343937047931), (23, 336.15510516737066), (24, 307.8503096701882), (25, 293.9176194566061), (26, 365.2022418913773), (27, 338.10957360080556), (28, 332.22855193653743), (29, 270.20487260298296), (30, 2

FedProx: {'method': 'fedprox', 'dataset': 'FD003', 'seed': 202, 'test_mae': 15.6629, 'nasa_score': 2142.73, 'comm_kb': 28900.78}


In [9]:
from run_experiment import run_simulation
print("Running FedProx")
fedprox_result = run_simulation('fedprox', 'FD003', 303)
print("FedProx:", fedprox_result)

Running FedProx

-- K-Means Clustering Results --
  - Cluster 0 assigned 44 engines.
  - Cluster 1 assigned 56 engines.
---------------------------------
✅ Created sequences: X shape = (4898, 30, 24), y shape = (4898,)
✅ Created sequences: X shape = (1446, 30, 24), y shape = (1446,)
✅ Created sequences: X shape = (4804, 30, 24), y shape = (4804,)
✅ Created sequences: X shape = (981, 30, 24), y shape = (981,)
✅ Created sequences: X shape = (4097, 30, 24), y shape = (4097,)
✅ Created sequences: X shape = (1142, 30, 24), y shape = (1142,)
✅ Created sequences: X shape = (3571, 30, 24), y shape = (3571,)
✅ Created sequences: X shape = (881, 30, 24), y shape = (881,)


INFO flwr 2026-07-13 08:46:00,564 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-13 08:46:11,690	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-13 08:46:15,638 | app.py:210 | Flower VCE: Ray initialized with resources: {'GPU': 2.0, 'memory': 15110556877.0, 'node:172.19.2.2': 1.0, 'CPU': 4.0, 'accelerator_type:T4': 1.0, 'object_store_memory': 6475952947.0, 'node:__internal_head__': 1.0}
INFO flwr 2026-07-13 08:46:15,639 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-07-13 08:46:15,665 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-07-13 08:46:15,666 | server.py:89 | Initializing global parameters
INFO flwr 2026-07-13 08:46:15,666 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-07-13 08:46:15,667 | server.py:91 | Evaluating initial parameters
INFO flwr 2026-07-13 08:

  [Round 0] Test MAE: 73.6070 | NASA: 433025.45


(pid=165473) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
(pid=165473) E0000 00:00:1783932377.908622  165473 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
(pid=165473) E0000 00:00:1783932377.968298  165473 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
(pid=165473) W0000 00:00:1783932378.027086  165473 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
(pid=165473) W0000 00:00:1783932378.027136  165473 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
(pid=165473) W0000 00:00:1783932378.027142  165473 computation_placer.cc:177] computation placer already registered. Please check linkage and avo

  [Round 1] Test MAE: 26.2473 | NASA: 21713.05


...

  [Round 49] Test MAE: 11.7442 | NASA: 1203.74


DEBUG flwr 2026-07-13 09:14:21,869 | server.py:187 | evaluate_round 49 received 4 results and 0 failures
DEBUG flwr 2026-07-13 09:14:21,870 | server.py:222 | fit_round 50: strategy sampled 4 clients (out of 4)
DEBUG flwr 2026-07-13 09:14:49,770 | server.py:236 | fit_round 50 received 4 results and 0 failures
INFO flwr 2026-07-13 09:14:51,075 | server.py:125 | fit progress: (50, 0.0, {'mae': 11.362808113098145, 'nasa_score': 1371.4449238643097}, 1713.4983876309998)
DEBUG flwr 2026-07-13 09:14:51,077 | server.py:173 | evaluate_round 50: strategy sampled 4 clients (out of 4)


  [Round 50] Test MAE: 11.3628 | NASA: 1371.44


DEBUG flwr 2026-07-13 09:14:54,489 | server.py:187 | evaluate_round 50 received 4 results and 0 failures
INFO flwr 2026-07-13 09:14:54,490 | server.py:153 | FL finished in 1716.9129293819997
INFO flwr 2026-07-13 09:14:54,490 | app.py:225 | app_fit: losses_distributed [(1, 1015.786105970747), (2, 669.2588324043188), (3, 391.1278314894773), (4, 402.8553998762838), (5, 413.5545504657874), (6, 346.9899975071596), (7, 423.48194557790004), (8, 392.07318232504167), (9, 396.03451707818533), (10, 485.7244871400983), (11, 382.1352544548806), (12, 373.6385211044483), (13, 337.0644506218728), (14, 315.07387148739247), (15, 315.6327968228801), (16, 297.2388114466292), (17, 325.46580628127197), (18, 296.8914145240355), (19, 292.5012327970012), (20, 307.4598572703158), (21, 334.7432932718684), (22, 277.1239843441395), (23, 276.0239860346076), (24, 282.0921986440594), (25, 263.9061667247301), (26, 259.8472912117604), (27, 275.95514908008363), (28, 244.23869950883844), (29, 252.62396933909213), (30, 30

FedProx: {'method': 'fedprox', 'dataset': 'FD003', 'seed': 303, 'test_mae': 11.3628, 'nasa_score': 1371.44, 'comm_kb': 28900.78}


In [10]:
from run_experiment import run_simulation
print("Running FedProx")
fedprox_result = run_simulation('fedprox', 'FD003', 2026)
print("FedProx:", fedprox_result)

Running FedProx

-- K-Means Clustering Results --
  - Cluster 0 assigned 44 engines.
  - Cluster 1 assigned 56 engines.
---------------------------------
✅ Created sequences: X shape = (4737, 30, 24), y shape = (4737,)
✅ Created sequences: X shape = (1607, 30, 24), y shape = (1607,)
✅ Created sequences: X shape = (4658, 30, 24), y shape = (4658,)
✅ Created sequences: X shape = (1127, 30, 24), y shape = (1127,)
✅ Created sequences: X shape = (4213, 30, 24), y shape = (4213,)
✅ Created sequences: X shape = (1026, 30, 24), y shape = (1026,)
✅ Created sequences: X shape = (3544, 30, 24), y shape = (3544,)
✅ Created sequences: X shape = (908, 30, 24), y shape = (908,)


INFO flwr 2026-07-13 09:15:14,760 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-13 09:15:26,284	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-13 09:15:29,714 | app.py:210 | Flower VCE: Ray initialized with resources: {'node:172.19.2.2': 1.0, 'object_store_memory': 6486922444.0, 'GPU': 2.0, 'accelerator_type:T4': 1.0, 'memory': 15136152372.0, 'node:__internal_head__': 1.0, 'CPU': 4.0}
INFO flwr 2026-07-13 09:15:29,715 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-07-13 09:15:29,753 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-07-13 09:15:29,754 | server.py:89 | Initializing global parameters
INFO flwr 2026-07-13 09:15:29,755 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-07-13 09:15:29,755 | server.py:91 | Evaluating initial parameters
(pid=220201) WARNING: Al

  [Round 0] Test MAE: 74.7887 | NASA: 481312.47


(DefaultActor pid=220201) I0000 00:00:1783934142.330621  220201 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13606 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
(pid=220202) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR [repeated 3x across cluster]
(pid=220202) E0000 00:00:1783934132.595959  220202 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered [repeated 3x across cluster]
(pid=220202) E0000 00:00:1783934132.608018  220202 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered [repeated 3x across cluster]
(pid=220202) W0000 00:00:1783934132.649580  220202 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.

  [Round 1] Test MAE: 23.5370 | NASA: 1663.37


...

  [Round 49] Test MAE: 11.9415 | NASA: 459.74


DEBUG flwr 2026-07-13 09:42:01,205 | server.py:187 | evaluate_round 49 received 4 results and 0 failures
DEBUG flwr 2026-07-13 09:42:01,206 | server.py:222 | fit_round 50: strategy sampled 4 clients (out of 4)
DEBUG flwr 2026-07-13 09:42:32,032 | server.py:236 | fit_round 50 received 4 results and 0 failures
INFO flwr 2026-07-13 09:42:33,402 | server.py:125 | fit progress: (50, 0.0, {'mae': 12.197911701202393, 'nasa_score': 582.8868202656263}, 1621.5924821429999)
DEBUG flwr 2026-07-13 09:42:33,403 | server.py:173 | evaluate_round 50: strategy sampled 4 clients (out of 4)


  [Round 50] Test MAE: 12.1979 | NASA: 582.89


DEBUG flwr 2026-07-13 09:42:37,186 | server.py:187 | evaluate_round 50 received 4 results and 0 failures
INFO flwr 2026-07-13 09:42:37,188 | server.py:153 | FL finished in 1625.3785887489994
INFO flwr 2026-07-13 09:42:37,189 | app.py:225 | app_fit: losses_distributed [(1, 1459.435808876384), (2, 368.7293873732174), (3, 422.91183122571147), (4, 413.6352289521807), (5, 476.56490678958846), (6, 438.6038275410604), (7, 492.7259495595427), (8, 453.2787475324433), (9, 458.6339311509839), (10, 404.9004432200977), (11, 570.1166863265769), (12, 524.1508389091329), (13, 412.6022001657102), (14, 361.3113107427942), (15, 328.97074111180115), (16, 402.725746952374), (17, 367.9210431998677), (18, 374.7113166946372), (19, 415.4033164226202), (20, 453.7234654299908), (21, 340.53589326454824), (22, 381.546149946015), (23, 336.1436539219298), (24, 387.8623145200702), (25, 375.522781022308), (26, 325.48166688201565), (27, 363.99910396128104), (28, 389.231691244431), (29, 421.0699530424169), (30, 318.3264

FedProx: {'method': 'fedprox', 'dataset': 'FD003', 'seed': 2026, 'test_mae': 12.1979, 'nasa_score': 582.89, 'comm_kb': 28900.78}


In [11]:
from run_experiment import run_simulation
print("Running Ditto (cluster-routed)...")
ditto_result = run_simulation('ditto', 'FD003', 303)
print("Ditto:", ditto_result)

Running Ditto (cluster-routed)...

-- K-Means Clustering Results --
  - Cluster 0 assigned 44 engines.
  - Cluster 1 assigned 56 engines.
---------------------------------
✅ Created sequences: X shape = (4898, 30, 24), y shape = (4898,)
✅ Created sequences: X shape = (1446, 30, 24), y shape = (1446,)
✅ Created sequences: X shape = (4804, 30, 24), y shape = (4804,)
✅ Created sequences: X shape = (981, 30, 24), y shape = (981,)
✅ Created sequences: X shape = (4097, 30, 24), y shape = (4097,)
✅ Created sequences: X shape = (1142, 30, 24), y shape = (1142,)
✅ Created sequences: X shape = (3571, 30, 24), y shape = (3571,)
✅ Created sequences: X shape = (881, 30, 24), y shape = (881,)


INFO flwr 2026-07-13 09:42:57,856 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-13 09:43:09,119	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-13 09:43:12,651 | app.py:210 | Flower VCE: Ray initialized with resources: {'node:__internal_head__': 1.0, 'CPU': 4.0, 'object_store_memory': 6465471283.0, 'memory': 15086099661.0, 'node:172.19.2.2': 1.0, 'accelerator_type:T4': 1.0, 'GPU': 2.0}
INFO flwr 2026-07-13 09:43:12,653 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-07-13 09:43:12,683 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-07-13 09:43:12,687 | server.py:89 | Initializing global parameters
INFO flwr 2026-07-13 09:43:12,693 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-07-13 09:43:12,694 | server.py:91 | Evaluating initial parameters
INFO flwr 2026-07-13 09:

Ditto: {'method': 'ditto', 'dataset': 'FD003', 'seed': 303, 'test_mae': 14.7543, 'nasa_score': 12753.74, 'comm_kb': 28900.78}


In [12]:
from run_experiment import run_simulation
print("Running Ditto (cluster-routed)...")
ditto_result = run_simulation('ditto', 'FD003', 42)
print("Ditto:", ditto_result)

Running Ditto (cluster-routed)...

-- K-Means Clustering Results --
  - Cluster 0 assigned 44 engines.
  - Cluster 1 assigned 56 engines.
---------------------------------
✅ Created sequences: X shape = (5028, 30, 24), y shape = (5028,)
✅ Created sequences: X shape = (1316, 30, 24), y shape = (1316,)
✅ Created sequences: X shape = (4687, 30, 24), y shape = (4687,)
✅ Created sequences: X shape = (1098, 30, 24), y shape = (1098,)
✅ Created sequences: X shape = (4203, 30, 24), y shape = (4203,)
✅ Created sequences: X shape = (1036, 30, 24), y shape = (1036,)
✅ Created sequences: X shape = (3515, 30, 24), y shape = (3515,)
✅ Created sequences: X shape = (937, 30, 24), y shape = (937,)


INFO flwr 2026-07-13 10:34:02,630 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-13 10:34:14,215	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-13 10:34:17,701 | app.py:210 | Flower VCE: Ray initialized with resources: {'CPU': 4.0, 'GPU': 2.0, 'node:172.19.2.2': 1.0, 'memory': 11557175706.0, 'object_store_memory': 4953075302.0, 'accelerator_type:T4': 1.0, 'node:__internal_head__': 1.0}
INFO flwr 2026-07-13 10:34:17,702 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-07-13 10:34:17,733 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-07-13 10:34:17,734 | server.py:89 | Initializing global parameters
INFO flwr 2026-07-13 10:34:17,739 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-07-13 10:34:17,741 | server.py:91 | Evaluating initial parameters
INFO flwr 2026-07-13 10:

Ditto: {'method': 'ditto', 'dataset': 'FD003', 'seed': 42, 'test_mae': 15.1741, 'nasa_score': 14287.33, 'comm_kb': 28900.78}


In [13]:
from run_experiment import run_simulation
print("Running Ditto (cluster-routed)...")
ditto_result = run_simulation('ditto', 'FD003', 101)
print("Ditto:", ditto_result)

Running Ditto (cluster-routed)...

-- K-Means Clustering Results --
  - Cluster 0 assigned 44 engines.
  - Cluster 1 assigned 56 engines.
---------------------------------
✅ Created sequences: X shape = (5107, 30, 24), y shape = (5107,)
✅ Created sequences: X shape = (1237, 30, 24), y shape = (1237,)
✅ Created sequences: X shape = (4174, 30, 24), y shape = (4174,)
✅ Created sequences: X shape = (1611, 30, 24), y shape = (1611,)
✅ Created sequences: X shape = (4214, 30, 24), y shape = (4214,)
✅ Created sequences: X shape = (1025, 30, 24), y shape = (1025,)
✅ Created sequences: X shape = (3513, 30, 24), y shape = (3513,)
✅ Created sequences: X shape = (939, 30, 24), y shape = (939,)


INFO flwr 2026-07-13 11:25:03,648 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-13 11:25:15,212	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-13 11:25:18,744 | app.py:210 | Flower VCE: Ray initialized with resources: {'accelerator_type:T4': 1.0, 'object_store_memory': 4937349120.0, 'memory': 11520481280.0, 'CPU': 4.0, 'node:172.19.2.2': 1.0, 'GPU': 2.0, 'node:__internal_head__': 1.0}
INFO flwr 2026-07-13 11:25:18,745 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-07-13 11:25:18,774 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-07-13 11:25:18,774 | server.py:89 | Initializing global parameters
INFO flwr 2026-07-13 11:25:18,775 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-07-13 11:25:18,776 | server.py:91 | Evaluating initial parameters
INFO flwr 2026-07-13 11:

Ditto: {'method': 'ditto', 'dataset': 'FD003', 'seed': 101, 'test_mae': 14.4631, 'nasa_score': 10487.81, 'comm_kb': 28900.78}


In [14]:
from run_experiment import run_simulation
print("Running Ditto (cluster-routed)...")
ditto_result = run_simulation('ditto', 'FD003', 202)
print("Ditto:", ditto_result)

Running Ditto (cluster-routed)...

-- K-Means Clustering Results --
  - Cluster 0 assigned 44 engines.
  - Cluster 1 assigned 56 engines.
---------------------------------
✅ Created sequences: X shape = (4485, 30, 24), y shape = (4485,)
✅ Created sequences: X shape = (1859, 30, 24), y shape = (1859,)
✅ Created sequences: X shape = (4383, 30, 24), y shape = (4383,)
✅ Created sequences: X shape = (1402, 30, 24), y shape = (1402,)
✅ Created sequences: X shape = (4176, 30, 24), y shape = (4176,)
✅ Created sequences: X shape = (1063, 30, 24), y shape = (1063,)
✅ Created sequences: X shape = (3551, 30, 24), y shape = (3551,)
✅ Created sequences: X shape = (901, 30, 24), y shape = (901,)


INFO flwr 2026-07-13 12:15:06,430 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-13 12:15:18,117	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-13 12:15:21,629 | app.py:210 | Flower VCE: Ray initialized with resources: {'accelerator_type:T4': 1.0, 'CPU': 4.0, 'node:__internal_head__': 1.0, 'node:172.19.2.2': 1.0, 'GPU': 2.0, 'object_store_memory': 4947007488.0, 'memory': 11543017472.0}
INFO flwr 2026-07-13 12:15:21,630 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-07-13 12:15:21,657 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-07-13 12:15:21,658 | server.py:89 | Initializing global parameters
INFO flwr 2026-07-13 12:15:21,661 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-07-13 12:15:21,662 | server.py:91 | Evaluating initial parameters
INFO flwr 2026-07-13 12:

Ditto: {'method': 'ditto', 'dataset': 'FD003', 'seed': 202, 'test_mae': 13.3647, 'nasa_score': 4377.65, 'comm_kb': 28900.78}


In [15]:
from run_experiment import run_simulation
print("Running Ditto (cluster-routed)...")
ditto_result = run_simulation('ditto', 'FD003', 2026)
print("Ditto:", ditto_result)

Running Ditto (cluster-routed)...

-- K-Means Clustering Results --
  - Cluster 0 assigned 44 engines.
  - Cluster 1 assigned 56 engines.
---------------------------------
✅ Created sequences: X shape = (4737, 30, 24), y shape = (4737,)
✅ Created sequences: X shape = (1607, 30, 24), y shape = (1607,)
✅ Created sequences: X shape = (4658, 30, 24), y shape = (4658,)
✅ Created sequences: X shape = (1127, 30, 24), y shape = (1127,)
✅ Created sequences: X shape = (4213, 30, 24), y shape = (4213,)
✅ Created sequences: X shape = (1026, 30, 24), y shape = (1026,)
✅ Created sequences: X shape = (3544, 30, 24), y shape = (3544,)
✅ Created sequences: X shape = (908, 30, 24), y shape = (908,)


INFO flwr 2026-07-13 13:05:32,904 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-13 13:05:41,205	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-13 13:05:44,538 | app.py:210 | Flower VCE: Ray initialized with resources: {'object_store_memory': 9283047014.0, 'accelerator_type:T4': 1.0, 'node:172.19.2.2': 1.0, 'GPU': 2.0, 'node:__internal_head__': 1.0, 'CPU': 4.0, 'memory': 21660443034.0}
INFO flwr 2026-07-13 13:05:44,539 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-07-13 13:05:44,557 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-07-13 13:05:44,559 | server.py:89 | Initializing global parameters
INFO flwr 2026-07-13 13:05:44,559 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-07-13 13:05:44,560 | server.py:91 | Evaluating initial parameters
INFO flwr 2026-07-13 13:

Ditto: {'method': 'ditto', 'dataset': 'FD003', 'seed': 2026, 'test_mae': 13.6579, 'nasa_score': 3348.61, 'comm_kb': 28900.78}


In [6]:
from run_experiment import run_simulation
print("Checking FedAvg seed 42...")
result = run_simulation('fedavg', 'FD002', 42)
print("FedAvg 42:", result)

E0000 00:00:1783961906.119010     115 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1783961906.173816     115 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1783961906.605857     115 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783961906.605899     115 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783961906.605902     115 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783961906.605904     115 computation_placer.cc:177] computation placer already registered. Please check linka

Checking FedAvg seed 42...

-- K-Means Clustering Results --
  - Cluster 0 assigned 129 engines.
  - Cluster 1 assigned 131 engines.
---------------------------------
✅ Created sequences: X shape = (8805, 30, 24), y shape = (8805,)
✅ Created sequences: X shape = (2345, 30, 24), y shape = (2345,)
✅ Created sequences: X shape = (9138, 30, 24), y shape = (9138,)
✅ Created sequences: X shape = (2270, 30, 24), y shape = (2270,)
✅ Created sequences: X shape = (9413, 30, 24), y shape = (9413,)
✅ Created sequences: X shape = (2582, 30, 24), y shape = (2582,)
✅ Created sequences: X shape = (9283, 30, 24), y shape = (9283,)
✅ Created sequences: X shape = (2383, 30, 24), y shape = (2383,)


I0000 00:00:1783961961.353507     115 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1783961961.359315     115 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
INFO flwr 2026-07-13 16:59:23,123 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-13 16:59:31,081	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-13 16:59:34,552 | app.py:210 | Flower VCE: Ray initialized with resources: {'GPU': 2.0, 'node:__internal_head__': 1.0, 'accelerator_type:T4': 1.0, 'CPU': 4.0, 'memory': 21577294234.0, 'object_store_memory': 9247411814.0, 'node:172.19.2.2': 1.0}
INFO flwr 2026-07-13 16:59:34,554 | app.py:224 | Flower VCE: Resources for each Virtu

  [Round 0] Test MAE: 74.3789 | NASA: 1484512.52


(DefaultActor pid=366) I0000 00:00:1783961985.682257     366 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13644 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
(pid=366) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR [repeated 3x across cluster] (Ray deduplicates logs by default. Set RAY_DEDUP_LOGS=0 to disable log deduplication, or see https://docs.ray.io/en/master/ray-observability/user-guides/configure-logging.html#log-deduplication for more options.)
(pid=366) E0000 00:00:1783961975.734944     366 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered [repeated 3x across cluster]
(pid=366) E0000 00:00:1783961975.750650     366 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered [repeated 3x ac

...

  [Round 49] Test MAE: 20.9786 | NASA: 25352.03


DEBUG flwr 2026-07-13 17:44:30,939 | server.py:187 | evaluate_round 49 received 4 results and 0 failures
DEBUG flwr 2026-07-13 17:44:30,940 | server.py:222 | fit_round 50: strategy sampled 4 clients (out of 4)
DEBUG flwr 2026-07-13 17:45:23,496 | server.py:236 | fit_round 50 received 4 results and 0 failures
INFO flwr 2026-07-13 17:45:24,904 | server.py:125 | fit progress: (50, 0.0, {'mae': 20.84194077303971, 'nasa_score': 8802.93590802131}, 2746.560878051)
DEBUG flwr 2026-07-13 17:45:24,905 | server.py:173 | evaluate_round 50: strategy sampled 4 clients (out of 4)


  [Round 50] Test MAE: 20.8419 | NASA: 8802.94


DEBUG flwr 2026-07-13 17:45:28,530 | server.py:187 | evaluate_round 50 received 4 results and 0 failures
INFO flwr 2026-07-13 17:45:28,531 | server.py:153 | FL finished in 2750.1880318149997
INFO flwr 2026-07-13 17:45:28,532 | app.py:225 | app_fit: losses_distributed [(1, 2101.669140930813), (2, 638.959560303499), (3, 533.5405302967557), (4, 529.3842740275915), (5, 531.6077291851004), (6, 530.5091882375187), (7, 573.7993239910468), (8, 580.1744831125025), (9, 604.4070563903681), (10, 638.9143298123226), (11, 633.4850274072062), (12, 676.5912641043454), (13, 648.699795430289), (14, 678.096426856692), (15, 660.203405647039), (16, 677.1562543259782), (17, 706.6835475595112), (18, 663.5346392605649), (19, 674.3562491462724), (20, 649.0938207383444), (21, 673.6680536767884), (22, 654.28665266256), (23, 664.6504710729038), (24, 686.4283003518377), (25, 690.6028784065007), (26, 684.259528358794), (27, 672.6456178987698), (28, 670.9626537729156), (29, 682.937276489003), (30, 686.5860364113571)

FedAvg 42: {'method': 'fedavg', 'dataset': 'FD002', 'seed': 42, 'test_mae': 20.8419, 'nasa_score': 8802.94, 'comm_kb': 28900.78}


In [ ]:
from run_experiment import run_simulation
print("Checking FedAvg seed 101...")
result = run_simulation('fedavg', 'FD002', 101)
print("FedAvg 101:", result)

Checking FedAvg seed 101...

-- K-Means Clustering Results --
  - Cluster 0 assigned 129 engines.
  - Cluster 1 assigned 131 engines.
---------------------------------
✅ Created sequences: X shape = (8952, 30, 24), y shape = (8952,)
✅ Created sequences: X shape = (2198, 30, 24), y shape = (2198,)
✅ Created sequences: X shape = (9044, 30, 24), y shape = (9044,)
✅ Created sequences: X shape = (2364, 30, 24), y shape = (2364,)
✅ Created sequences: X shape = (9526, 30, 24), y shape = (9526,)
✅ Created sequences: X shape = (2469, 30, 24), y shape = (2469,)
✅ Created sequences: X shape = (9212, 30, 24), y shape = (9212,)
✅ Created sequences: X shape = (2454, 30, 24), y shape = (2454,)


INFO flwr 2026-07-13 17:46:09,507 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=50, round_timeout=None)
2026-07-13 17:46:22,440	INFO worker.py:2012 -- Started a local Ray instance.
INFO flwr 2026-07-13 17:46:25,985 | app.py:210 | Flower VCE: Ray initialized with resources: {'node:__internal_head__': 1.0, 'GPU': 2.0, 'node:172.19.2.2': 1.0, 'object_store_memory': 6557242982.0, 'accelerator_type:T4': 1.0, 'memory': 15300233626.0, 'CPU': 4.0}
INFO flwr 2026-07-13 17:46:25,986 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.5}
INFO flwr 2026-07-13 17:46:26,011 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 4 actors
INFO flwr 2026-07-13 17:46:26,012 | server.py:89 | Initializing global parameters
INFO flwr 2026-07-13 17:46:26,013 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-07-13 17:46:26,014 | server.py:91 | Evaluating initial parameters
INFO flwr 2026-07-13 17:

  [Round 0] Test MAE: 75.5377 | NASA: 1622879.91


(pid=55197) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
(pid=55197) E0000 00:00:1783964788.093567   55197 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
(pid=55197) E0000 00:00:1783964788.106574   55197 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
(pid=55197) W0000 00:00:1783964788.140028   55197 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
(pid=55197) W0000 00:00:1783964788.140089   55197 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
(pid=55197) W0000 00:00:1783964788.140095   55197 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid lin

In [6]:
from run_experiment import run_acpfl
result = run_acpfl('FD004', 42, num_clusters=2, enable_reclustering=False)
print("AC-PFL Static:", result)

E0000 00:00:1784815480.624121     116 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1784815480.691143     116 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1784815481.302125     116 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784815481.302166     116 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784815481.302169     116 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784815481.302171     116 computation_placer.cc:177] computation placer already registered. Please check linka


-- K-Means Clustering Results --
  - Cluster 0 assigned 132 engines.
  - Cluster 1 assigned 117 engines.
---------------------------------
✅ Created sequences: X shape = (12060, 30, 24), y shape = (12060,)
✅ Created sequences: X shape = (2448, 30, 24), y shape = (2448,)
✅ Created sequences: X shape = (11660, 30, 24), y shape = (11660,)
✅ Created sequences: X shape = (2846, 30, 24), y shape = (2846,)
✅ Created sequences: X shape = (10505, 30, 24), y shape = (10505,)
✅ Created sequences: X shape = (2401, 30, 24), y shape = (2401,)
✅ Created sequences: X shape = (9866, 30, 24), y shape = (9866,)
✅ Created sequences: X shape = (2242, 30, 24), y shape = (2242,)


I0000 00:00:1784815545.524894     116 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1784815545.530773     116 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1784815552.905132     163 cuda_dnn.cc:529] Loaded cuDNN version 91002


  [Round 1] Val NASA per client: [79927.3, 46595.2, 49912.3, 24965.0]
  [Round 2] Val NASA per client: [73035.7, 43539.0, 34322.8, 25564.6]
  [Round 3] Val NASA per client: [52880.8, 53276.8, 30766.2, 17096.4]
  [Round 4] Val NASA per client: [56982.6, 55932.9, 29846.1, 30171.0]
  [Round 5] Val NASA per client: [77411.5, 41341.3, 30322.7, 18703.1]
  [Round 6] Val NASA per client: [62146.4, 52481.7, 30357.3, 19479.4]
  [Round 7] Val NASA per client: [54494.0, 40101.2, 32799.5, 19319.5]
  [Round 8] Val NASA per client: [43395.1, 36486.9, 36455.3, 19750.8]
  [Round 9] Val NASA per client: [68909.5, 50894.3, 39079.6, 23558.6]
  [Round 10] Val NASA per client: [69887.9, 55376.7, 40587.0, 21189.8]
  [Round 11] Val NASA per client: [62464.9, 52285.1, 52212.2, 27869.6]
  [Round 12] Val NASA per client: [52874.4, 45796.2, 38340.8, 29221.9]
  [Round 13] Val NASA per client: [54181.1, 59185.9, 29071.0, 35793.0]
  [Round 14] Val NASA per client: [99356.0, 51744.0, 42650.6, 41023.4]
  [Round 15] Va

In [6]:
from run_experiment import run_acpfl
result = run_acpfl('FD004', 101, num_clusters=2, enable_reclustering=False)
print("AC-PFL Static:", result)

E0000 00:00:1786022408.909235     115 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1786022408.977554     115 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1786022409.597473     115 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1786022409.597508     115 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1786022409.597511     115 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1786022409.597513     115 computation_placer.cc:177] computation placer already registered. Please check linka


-- K-Means Clustering Results --
  - Cluster 0 assigned 132 engines.
  - Cluster 1 assigned 117 engines.
---------------------------------
✅ Created sequences: X shape = (11353, 30, 24), y shape = (11353,)
✅ Created sequences: X shape = (3155, 30, 24), y shape = (3155,)
✅ Created sequences: X shape = (11101, 30, 24), y shape = (11101,)
✅ Created sequences: X shape = (3405, 30, 24), y shape = (3405,)
✅ Created sequences: X shape = (10244, 30, 24), y shape = (10244,)
✅ Created sequences: X shape = (2662, 30, 24), y shape = (2662,)
✅ Created sequences: X shape = (9748, 30, 24), y shape = (9748,)
✅ Created sequences: X shape = (2360, 30, 24), y shape = (2360,)


I0000 00:00:1786022474.108739     115 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1786022474.114937     115 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1786022481.475200     167 cuda_dnn.cc:529] Loaded cuDNN version 91002


  [Round 1] Val NASA per client: [475079.9, 443566.1, 55355.2, 62763.4]
  [Round 2] Val NASA per client: [73957.7, 47026.1, 43086.2, 69892.0]
  [Round 3] Val NASA per client: [64337.0, 47340.1, 37669.1, 27115.8]
  [Round 4] Val NASA per client: [51328.6, 42547.3, 28970.8, 28358.1]
  [Round 5] Val NASA per client: [70219.1, 42170.3, 39920.0, 36559.5]
  [Round 6] Val NASA per client: [52455.9, 38054.7, 35167.3, 43207.1]
  [Round 7] Val NASA per client: [35741.7, 55624.2, 37905.2, 34645.0]
  [Round 8] Val NASA per client: [37918.2, 48457.4, 39773.6, 30946.7]
  [Round 9] Val NASA per client: [29320.0, 36672.3, 31447.5, 36125.0]
  [Round 10] Val NASA per client: [37402.1, 44607.0, 44159.8, 43025.3]
  [Round 11] Val NASA per client: [43526.6, 42017.5, 37631.5, 44927.1]
  [Round 12] Val NASA per client: [104928.3, 46588.9, 39987.7, 37161.7]
  [Round 13] Val NASA per client: [44417.2, 40690.1, 51969.5, 35080.2]
  [Round 14] Val NASA per client: [38649.8, 62043.4, 51920.9, 34261.4]
  [Round 15]

In [6]:
from run_experiment import run_acpfl
result = run_acpfl('FD004', 202, num_clusters=2, enable_reclustering=False)
print("AC-PFL Static:", result)

E0000 00:00:1786081034.267397     115 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1786081034.322543     115 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1786081034.705609     115 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1786081034.705646     115 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1786081034.705649     115 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1786081034.705651     115 computation_placer.cc:177] computation placer already registered. Please check linka


-- K-Means Clustering Results --
  - Cluster 0 assigned 132 engines.
  - Cluster 1 assigned 117 engines.
---------------------------------
✅ Created sequences: X shape = (11720, 30, 24), y shape = (11720,)
✅ Created sequences: X shape = (2788, 30, 24), y shape = (2788,)
✅ Created sequences: X shape = (11430, 30, 24), y shape = (11430,)
✅ Created sequences: X shape = (3076, 30, 24), y shape = (3076,)
✅ Created sequences: X shape = (10465, 30, 24), y shape = (10465,)
✅ Created sequences: X shape = (2441, 30, 24), y shape = (2441,)
✅ Created sequences: X shape = (9809, 30, 24), y shape = (9809,)
✅ Created sequences: X shape = (2299, 30, 24), y shape = (2299,)


I0000 00:00:1786081098.692832     115 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1786081098.698557     115 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1786081105.776153     162 cuda_dnn.cc:529] Loaded cuDNN version 91002


  [Round 1] Val NASA per client: [126626.4, 156634.5, 94216.7, 39325.9]
  [Round 2] Val NASA per client: [109013.2, 218381.6, 59973.1, 25576.7]
  [Round 3] Val NASA per client: [224187.2, 142972.9, 45723.4, 45301.6]
  [Round 4] Val NASA per client: [52847.1, 154308.7, 60040.1, 57282.5]
  [Round 5] Val NASA per client: [55772.5, 182981.2, 77858.4, 37636.7]
  [Round 6] Val NASA per client: [47318.4, 205436.7, 47368.5, 33169.7]
  [Round 7] Val NASA per client: [66179.7, 224979.3, 57133.8, 45864.7]
  [Round 8] Val NASA per client: [49533.8, 250392.4, 45882.0, 39102.7]
  [Round 9] Val NASA per client: [83612.4, 245648.7, 39697.7, 42240.7]
  [Round 10] Val NASA per client: [52196.8, 232779.7, 53802.2, 49238.6]
  [Round 11] Val NASA per client: [61449.1, 254315.7, 64624.0, 78568.8]
  [Round 12] Val NASA per client: [88639.1, 220476.9, 60098.0, 49676.1]
  [Round 13] Val NASA per client: [99181.0, 242200.3, 64135.7, 84886.8]
  [Round 14] Val NASA per client: [83888.3, 201965.2, 59972.9, 70871.5

In [6]:
from run_experiment import run_acpfl
result = run_acpfl('FD004', 303, num_clusters=2, enable_reclustering=False)
print("AC-PFL Static:", result)

E0000 00:00:1786087808.487899     116 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1786087808.582240     116 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1786087809.542642     116 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1786087809.542684     116 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1786087809.542686     116 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1786087809.542689     116 computation_placer.cc:177] computation placer already registered. Please check linka


-- K-Means Clustering Results --
  - Cluster 0 assigned 132 engines.
  - Cluster 1 assigned 117 engines.
---------------------------------
✅ Created sequences: X shape = (11380, 30, 24), y shape = (11380,)
✅ Created sequences: X shape = (3128, 30, 24), y shape = (3128,)
✅ Created sequences: X shape = (10903, 30, 24), y shape = (10903,)
✅ Created sequences: X shape = (3603, 30, 24), y shape = (3603,)
✅ Created sequences: X shape = (10533, 30, 24), y shape = (10533,)
✅ Created sequences: X shape = (2373, 30, 24), y shape = (2373,)
✅ Created sequences: X shape = (9746, 30, 24), y shape = (9746,)
✅ Created sequences: X shape = (2362, 30, 24), y shape = (2362,)


I0000 00:00:1786087874.800602     116 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1786087874.806192     116 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1786087882.321542     165 cuda_dnn.cc:529] Loaded cuDNN version 91002


  [Round 1] Val NASA per client: [86254.2, 136029.5, 67270.1, 55788.7]
  [Round 2] Val NASA per client: [64735.7, 75796.7, 57759.9, 55263.4]
  [Round 3] Val NASA per client: [48367.3, 62881.7, 50015.2, 45544.9]
  [Round 4] Val NASA per client: [68922.6, 82833.1, 72682.8, 39700.6]
  [Round 5] Val NASA per client: [73249.5, 36446.7, 63747.5, 45578.1]
  [Round 6] Val NASA per client: [56199.2, 37640.6, 43165.9, 44396.9]
  [Round 7] Val NASA per client: [67779.9, 30874.2, 37802.2, 35811.7]
  [Round 8] Val NASA per client: [41171.1, 32271.9, 32502.4, 42611.6]
  [Round 9] Val NASA per client: [54255.8, 32282.1, 26991.0, 44237.2]
  [Round 10] Val NASA per client: [32905.8, 33636.3, 37268.4, 32320.4]
  [Round 11] Val NASA per client: [96897.7, 38858.0, 33241.8, 34730.4]
  [Round 12] Val NASA per client: [98615.4, 44949.9, 37984.4, 41655.8]
  [Round 40] Val NASA per client: [106000.3, 116479.7, 84057.3, 64453.6]
  [Round 41] Val NASA per client: [81447.4, 107321.5, 88965.0, 67459.7]
  [Round 42

In [7]:
from run_experiment import run_acpfl
result = run_acpfl('FD004', 2026, num_clusters=2, enable_reclustering=False)
print("AC-PFL Static:", result)


-- K-Means Clustering Results --
  - Cluster 0 assigned 132 engines.
  - Cluster 1 assigned 117 engines.
---------------------------------
✅ Created sequences: X shape = (11559, 30, 24), y shape = (11559,)
✅ Created sequences: X shape = (2949, 30, 24), y shape = (2949,)
✅ Created sequences: X shape = (11576, 30, 24), y shape = (11576,)
✅ Created sequences: X shape = (2930, 30, 24), y shape = (2930,)
✅ Created sequences: X shape = (10377, 30, 24), y shape = (10377,)
✅ Created sequences: X shape = (2529, 30, 24), y shape = (2529,)
✅ Created sequences: X shape = (9810, 30, 24), y shape = (9810,)
✅ Created sequences: X shape = (2298, 30, 24), y shape = (2298,)
  [Round 1] Val NASA per client: [570258.8, 199795.5, 245055.1, 257180.3]
  [Round 2] Val NASA per client: [54841.4, 146639.6, 26817.5, 63398.6]
  [Round 3] Val NASA per client: [28044.6, 238736.4, 35818.1, 58512.6]
  [Round 4] Val NASA per client: [32665.4, 50016.8, 54313.1, 81168.2]
  [Round 5] Val NASA per client: [28304.1, 82203

In [6]:
from run_experiment import run_acpfl
result = run_acpfl('FD004', 42, num_clusters=2, share_heads=True)
print("AC-PFL SharedHeads:", result)

E0000 00:00:1786166841.840517     113 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1786166841.901971     113 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1786166842.442041     113 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1786166842.442088     113 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1786166842.442091     113 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1786166842.442093     113 computation_placer.cc:177] computation placer already registered. Please check linka


-- K-Means Clustering Results --
  - Cluster 0 assigned 132 engines.
  - Cluster 1 assigned 117 engines.
---------------------------------
✅ Created sequences: X shape = (12060, 30, 24), y shape = (12060,)
✅ Created sequences: X shape = (2448, 30, 24), y shape = (2448,)
✅ Created sequences: X shape = (11660, 30, 24), y shape = (11660,)
✅ Created sequences: X shape = (2846, 30, 24), y shape = (2846,)
✅ Created sequences: X shape = (10505, 30, 24), y shape = (10505,)
✅ Created sequences: X shape = (2401, 30, 24), y shape = (2401,)
✅ Created sequences: X shape = (9866, 30, 24), y shape = (9866,)
✅ Created sequences: X shape = (2242, 30, 24), y shape = (2242,)


I0000 00:00:1786166911.250753     113 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1786166911.256946     113 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1786166919.158075     164 cuda_dnn.cc:529] Loaded cuDNN version 91002


  [Round 1] Val NASA per client: [452958.0, 75285.7, 52814.2, 51459.6]
  [Round 2] Val NASA per client: [262363.6, 57892.5, 55595.5, 15490.8]
  [Round 3] Val NASA per client: [245769.6, 51331.5, 33121.7, 16343.6]
  [Round 4] Val NASA per client: [117323.3, 81910.6, 35368.1, 20372.6]
  [Round 5] Re-clustered (α=1.0). Changes: {}
  [Round 5] Assignments: {'0': 0, '1': 0, '2': 1, '3': 1}
  [Round 5] Cluster 0 val NASA: 157762.26
  [Round 5] Cluster 1 val NASA: 24304.70
  [Round 5] Val NASA per client: [208845.1, 106679.5, 29458.7, 19150.7]
  [Round 6] Val NASA per client: [61390.9, 104812.0, 29473.1, 21808.9]
  [Round 7] Val NASA per client: [89014.7, 70585.1, 27155.2, 34538.8]
  [Round 8] Val NASA per client: [79923.7, 83223.0, 34688.8, 21860.6]
  [Round 9] Val NASA per client: [78622.7, 67939.4, 43253.0, 25878.8]
  ⚠️ Spectral produced invalid clusters {0: 3, 1: 1}. Falling back to KMeans.
  KMeans fallback result: {0: 3, 1: 1}
  [Round 10] Re-clustered (α=1.0). Changes: {2: (1, 0)}
  [

In [7]:
from run_experiment import run_acpfl
result = run_acpfl('FD004', 101, num_clusters=2, share_heads=True)
print("AC-PFL SharedHeads:", result)

  [Round 19] Val NASA per client: [83208.2, 73093.9, 51672.2, 34096.3]
  [Round 20] Re-clustered (α=1.0). Changes: {1: (0, 1), 3: (1, 0)}
  [Round 20] Assignments: {'0': 0, '1': 1, '2': 1, '3': 0}
  [Round 20] Cluster 0 val NASA: 39184.30
  [Round 20] Cluster 1 val NASA: 52325.80
  [Round 20] Val NASA per client: [42807.9, 50236.8, 54414.8, 35560.7]
  [Round 21] Val NASA per client: [111690.3, 56766.4, 57611.8, 17357.7]
  [Round 22] Val NASA per client: [48464.5, 46955.1, 50275.1, 18899.8]
  [Round 23] Val NASA per client: [53199.3, 58044.5, 44704.8, 26853.1]
  [Round 24] Val NASA per client: [50893.9, 79508.9, 66061.4, 35752.4]
  ⚠️ Spectral produced invalid clusters {0: 3, 1: 1}. Falling back to KMeans.
  KMeans fallback result: {0: 2, 1: 2}
  [Round 25] Re-clustered (α=1.0). Changes: {2: (1, 0), 3: (0, 1)}
  [Round 25] Assignments: {'0': 0, '1': 1, '2': 0, '3': 1}
  [Round 25] Cluster 0 val NASA: 44382.86
  [Round 25] Cluster 1 val NASA: 57352.33
  [Round 25] Val NASA per client: [4

In [7]:
from run_experiment import run_acpfl
result = run_acpfl('FD004', 202, num_clusters=2, share_heads=True)
print("AC-PFL SharedHeads:", results)


-- K-Means Clustering Results --
  - Cluster 0 assigned 132 engines.
  - Cluster 1 assigned 117 engines.
---------------------------------
✅ Created sequences: X shape = (11720, 30, 24), y shape = (11720,)
✅ Created sequences: X shape = (2788, 30, 24), y shape = (2788,)
✅ Created sequences: X shape = (11430, 30, 24), y shape = (11430,)
✅ Created sequences: X shape = (3076, 30, 24), y shape = (3076,)
✅ Created sequences: X shape = (10465, 30, 24), y shape = (10465,)
✅ Created sequences: X shape = (2441, 30, 24), y shape = (2441,)
✅ Created sequences: X shape = (9809, 30, 24), y shape = (9809,)
✅ Created sequences: X shape = (2299, 30, 24), y shape = (2299,)
  [Round 1] Val NASA per client: [202867.6, 211713.7, 69794.5, 434172.6]
  [Round 2] Val NASA per client: [165251.2, 180623.7, 54810.6, 48171.2]
  [Round 3] Val NASA per client: [278696.1, 188652.0, 46797.1, 36256.5]
  [Round 4] Val NASA per client: [78200.1, 165635.2, 166956.0, 29332.8]
  [Round 5] Re-clustered (α=1.0). Changes: {}

NameError: name 'results' is not defined

In [7]:
from run_experiment import run_acpfl
results = run_acpfl('FD004', 303, num_clusters=2, share_heads=True)
print("AC-PFL SharedHeads:", results)


-- K-Means Clustering Results --
  - Cluster 0 assigned 132 engines.
  - Cluster 1 assigned 117 engines.
---------------------------------
✅ Created sequences: X shape = (11380, 30, 24), y shape = (11380,)
✅ Created sequences: X shape = (3128, 30, 24), y shape = (3128,)
✅ Created sequences: X shape = (10903, 30, 24), y shape = (10903,)
✅ Created sequences: X shape = (3603, 30, 24), y shape = (3603,)
✅ Created sequences: X shape = (10533, 30, 24), y shape = (10533,)
✅ Created sequences: X shape = (2373, 30, 24), y shape = (2373,)
✅ Created sequences: X shape = (9746, 30, 24), y shape = (9746,)
✅ Created sequences: X shape = (2362, 30, 24), y shape = (2362,)


I0000 00:00:1786181990.266100     114 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1786181990.271923     114 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1786181997.323807     169 cuda_dnn.cc:529] Loaded cuDNN version 91002


  [Round 1] Val NASA per client: [114408.7, 48878.2, 70515.4, 89614.1]
  [Round 2] Val NASA per client: [72484.3, 49907.8, 82693.5, 62312.1]
  [Round 3] Val NASA per client: [126016.6, 46297.6, 72268.3, 78239.7]
  [Round 4] Val NASA per client: [65825.5, 52105.7, 43652.1, 45901.7]
  [Round 5] Re-clustered (α=1.0). Changes: {}
  [Round 5] Assignments: {'0': 0, '1': 0, '2': 1, '3': 1}
  [Round 5] Cluster 0 val NASA: 53373.15
  [Round 5] Cluster 1 val NASA: 55742.64
  [Round 5] Val NASA per client: [65958.7, 40787.6, 59579.2, 51906.1]
  [Round 6] Val NASA per client: [65265.3, 37863.0, 60072.0, 63065.3]
  [Round 7] Val NASA per client: [137734.1, 31667.7, 47240.9, 44970.8]
  [Round 8] Val NASA per client: [84379.8, 49331.4, 62529.1, 54009.6]
  [Round 9] Val NASA per client: [44977.4, 38226.4, 71535.3, 43796.6]
  ⚠️ Spectral produced invalid clusters {0: 3, 1: 1}. Falling back to KMeans.
  KMeans fallback result: {0: 2, 1: 2}
  [Round 10] Re-clustered (α=1.0). Changes: {1: (0, 1), 2: (1, 0

In [8]:
from run_experiment import run_acpfl
results = run_acpfl('FD004', 2026, num_clusters=2, share_heads=True)
print("AC-PFL SharedHeads:", results)


-- K-Means Clustering Results --
  - Cluster 0 assigned 132 engines.
  - Cluster 1 assigned 117 engines.
---------------------------------
✅ Created sequences: X shape = (11559, 30, 24), y shape = (11559,)
✅ Created sequences: X shape = (2949, 30, 24), y shape = (2949,)
✅ Created sequences: X shape = (11576, 30, 24), y shape = (11576,)
✅ Created sequences: X shape = (2930, 30, 24), y shape = (2930,)
✅ Created sequences: X shape = (10377, 30, 24), y shape = (10377,)
✅ Created sequences: X shape = (2529, 30, 24), y shape = (2529,)
✅ Created sequences: X shape = (9810, 30, 24), y shape = (9810,)
✅ Created sequences: X shape = (2298, 30, 24), y shape = (2298,)
  [Round 1] Val NASA per client: [21918.7, 137151.9, 55769.3, 296274.9]
  [Round 2] Val NASA per client: [31577.8, 77641.2, 43617.0, 87096.0]
  [Round 3] Val NASA per client: [32264.4, 93131.8, 46886.6, 76007.4]
  [Round 4] Val NASA per client: [18108.5, 83364.5, 48815.7, 50673.8]
  [Round 5] Re-clustered (α=1.0). Changes: {}
  [Rou

In [6]:
from run_experiment import run_acpfl
result = run_acpfl('FD001', 42, num_clusters=2, enable_reclustering=False)
print("AC-PFL Static:", result)

E0000 00:00:1786194615.511241     115 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1786194615.660064     115 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1786194616.753432     115 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1786194616.753478     115 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1786194616.753481     115 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1786194616.753483     115 computation_placer.cc:177] computation placer already registered. Please check linka


-- K-Means Clustering Results --
  - Cluster 0 assigned 54 engines.
  - Cluster 1 assigned 46 engines.
---------------------------------
✅ Created sequences: X shape = (3281, 30, 24), y shape = (3281,)
✅ Created sequences: X shape = (925, 30, 24), y shape = (925,)
✅ Created sequences: X shape = (3803, 30, 24), y shape = (3803,)
✅ Created sequences: X shape = (1151, 30, 24), y shape = (1151,)
✅ Created sequences: X shape = (3270, 30, 24), y shape = (3270,)
✅ Created sequences: X shape = (944, 30, 24), y shape = (944,)
✅ Created sequences: X shape = (3582, 30, 24), y shape = (3582,)
✅ Created sequences: X shape = (775, 30, 24), y shape = (775,)


I0000 00:00:1786194654.379495     115 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1786194654.382355     115 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1786194662.010705     166 cuda_dnn.cc:529] Loaded cuDNN version 91002


  [Round 1] Val NASA per client: [4229.0, 9325.0, 1796.2, 1564.1]
  [Round 2] Val NASA per client: [3057.1, 9697.4, 1220.3, 2021.0]
  [Round 3] Val NASA per client: [3282.3, 7211.6, 1374.2, 2353.2]
  [Round 4] Val NASA per client: [3070.1, 6426.3, 1364.2, 1878.1]
  [Round 5] Val NASA per client: [2779.8, 10537.0, 1378.1, 3241.9]
  [Round 6] Val NASA per client: [3027.6, 6804.6, 1061.8, 2272.2]
  [Round 7] Val NASA per client: [3192.8, 8262.7, 1511.7, 2659.8]
  [Round 8] Val NASA per client: [2914.0, 7094.8, 1164.3, 1683.0]
  [Round 9] Val NASA per client: [2968.9, 7480.7, 1593.2, 2270.4]
  [Round 10] Val NASA per client: [2765.9, 6104.3, 1461.7, 4610.2]
  [Round 11] Val NASA per client: [3865.0, 6518.4, 1232.9, 4501.0]
  [Round 12] Val NASA per client: [3044.9, 7798.2, 1425.8, 4093.4]
  [Round 13] Val NASA per client: [4524.0, 5535.7, 1255.3, 3691.8]
  [Round 14] Val NASA per client: [5739.0, 7129.1, 1267.4, 2791.0]
  [Round 15] Val NASA per client: [2741.5, 6079.5, 1477.3, 3964.2]
  [

In [6]:
from run_experiment import run_acpfl
result = run_acpfl('FD001', 101, num_clusters=2, enable_reclustering=False)
print("AC-PFL Static:", result)

E0000 00:00:1786379943.405288     113 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1786379943.475152     113 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1786379944.111387     113 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1786379944.111430     113 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1786379944.111433     113 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1786379944.111436     113 computation_placer.cc:177] computation placer already registered. Please check linka


-- K-Means Clustering Results --
  - Cluster 0 assigned 54 engines.
  - Cluster 1 assigned 46 engines.
---------------------------------
✅ Created sequences: X shape = (3249, 30, 24), y shape = (3249,)
✅ Created sequences: X shape = (957, 30, 24), y shape = (957,)
✅ Created sequences: X shape = (3588, 30, 24), y shape = (3588,)
✅ Created sequences: X shape = (1366, 30, 24), y shape = (1366,)
✅ Created sequences: X shape = (3317, 30, 24), y shape = (3317,)
✅ Created sequences: X shape = (897, 30, 24), y shape = (897,)
✅ Created sequences: X shape = (3575, 30, 24), y shape = (3575,)
✅ Created sequences: X shape = (782, 30, 24), y shape = (782,)


I0000 00:00:1786379976.877058     113 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1786379976.883205     113 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1786379984.109530     164 cuda_dnn.cc:529] Loaded cuDNN version 91002


  [Round 1] Val NASA per client: [4770.3, 13546.6, 3101.1, 2181.5]
  [Round 2] Val NASA per client: [3477.0, 14122.0, 2441.5, 1731.2]
  [Round 3] Val NASA per client: [3440.4, 14143.0, 2447.1, 1915.5]
  [Round 4] Val NASA per client: [2847.4, 12413.9, 2948.5, 2642.7]
  [Round 5] Val NASA per client: [2448.2, 13746.0, 2612.6, 1440.5]
  [Round 6] Val NASA per client: [2704.2, 11414.5, 2794.5, 2057.3]
  [Round 7] Val NASA per client: [2593.8, 18575.5, 2495.4, 1994.2]
  [Round 8] Val NASA per client: [3061.4, 10789.1, 2836.2, 2345.7]
  [Round 9] Val NASA per client: [3115.9, 12564.4, 2366.6, 1873.9]
  [Round 10] Val NASA per client: [2727.4, 15249.2, 3051.6, 2572.2]
  [Round 11] Val NASA per client: [2212.9, 12607.9, 3046.8, 2540.6]
  [Round 12] Val NASA per client: [2447.4, 14692.3, 2629.6, 2349.1]
  [Round 13] Val NASA per client: [3158.1, 11271.6, 2929.3, 1892.0]
  [Round 14] Val NASA per client: [2415.1, 12478.8, 2436.9, 2220.6]
  [Round 15] Val NASA per client: [2389.4, 11723.4, 3034.

In [7]:
from run_experiment import run_acpfl
result = run_acpfl('FD001', 202, num_clusters=2, enable_reclustering=False)
print("AC-PFL Static:", result)


-- K-Means Clustering Results --
  - Cluster 0 assigned 54 engines.
  - Cluster 1 assigned 46 engines.
---------------------------------
✅ Created sequences: X shape = (3228, 30, 24), y shape = (3228,)
✅ Created sequences: X shape = (978, 30, 24), y shape = (978,)
✅ Created sequences: X shape = (3704, 30, 24), y shape = (3704,)
✅ Created sequences: X shape = (1250, 30, 24), y shape = (1250,)
✅ Created sequences: X shape = (3407, 30, 24), y shape = (3407,)
✅ Created sequences: X shape = (807, 30, 24), y shape = (807,)
✅ Created sequences: X shape = (3378, 30, 24), y shape = (3378,)
✅ Created sequences: X shape = (979, 30, 24), y shape = (979,)
  [Round 1] Val NASA per client: [2668.5, 9149.9, 2163.8, 5153.1]
  [Round 2] Val NASA per client: [2423.0, 10021.5, 2240.0, 6677.9]
  [Round 3] Val NASA per client: [2776.8, 7998.7, 1817.8, 5921.7]
  [Round 4] Val NASA per client: [2192.5, 7810.7, 2021.0, 5327.9]
  [Round 5] Val NASA per client: [2871.7, 10250.5, 1943.7, 5610.9]
  [Round 6] Val 

In [8]:
from run_experiment import run_acpfl
result = run_acpfl('FD001', 303, num_clusters=2, enable_reclustering=False)
print("AC-PFL Static:", result)


-- K-Means Clustering Results --
  - Cluster 0 assigned 54 engines.
  - Cluster 1 assigned 46 engines.
---------------------------------
✅ Created sequences: X shape = (3238, 30, 24), y shape = (3238,)
✅ Created sequences: X shape = (968, 30, 24), y shape = (968,)
✅ Created sequences: X shape = (3950, 30, 24), y shape = (3950,)
✅ Created sequences: X shape = (1004, 30, 24), y shape = (1004,)
✅ Created sequences: X shape = (3321, 30, 24), y shape = (3321,)
✅ Created sequences: X shape = (893, 30, 24), y shape = (893,)
✅ Created sequences: X shape = (3460, 30, 24), y shape = (3460,)
✅ Created sequences: X shape = (897, 30, 24), y shape = (897,)
  [Round 1] Val NASA per client: [3650.3, 4708.3, 2104.6, 2084.4]
  [Round 2] Val NASA per client: [4541.8, 6139.7, 2110.9, 1806.2]
  [Round 3] Val NASA per client: [4046.2, 7080.9, 1851.5, 1698.8]
  [Round 4] Val NASA per client: [3542.3, 6802.7, 2090.6, 4566.1]
  [Round 5] Val NASA per client: [4189.8, 4949.8, 2060.1, 2555.0]
  [Round 6] Val NA

In [9]:
from run_experiment import run_acpfl
result = run_acpfl('FD001', 2026, num_clusters=2, enable_reclustering=False)
print("AC-PFL Static:", result)


-- K-Means Clustering Results --
  - Cluster 0 assigned 54 engines.
  - Cluster 1 assigned 46 engines.
---------------------------------
✅ Created sequences: X shape = (3350, 30, 24), y shape = (3350,)
✅ Created sequences: X shape = (856, 30, 24), y shape = (856,)
✅ Created sequences: X shape = (3729, 30, 24), y shape = (3729,)
✅ Created sequences: X shape = (1225, 30, 24), y shape = (1225,)
✅ Created sequences: X shape = (3346, 30, 24), y shape = (3346,)
✅ Created sequences: X shape = (868, 30, 24), y shape = (868,)
✅ Created sequences: X shape = (3312, 30, 24), y shape = (3312,)
✅ Created sequences: X shape = (1045, 30, 24), y shape = (1045,)
  [Round 1] Val NASA per client: [1637.2, 5511.2, 4492.6, 25645.9]
  [Round 2] Val NASA per client: [1556.4, 5359.2, 3689.3, 11580.3]
  [Round 3] Val NASA per client: [1646.0, 5526.1, 3978.7, 19063.7]
  [Round 4] Val NASA per client: [2010.0, 6791.1, 2770.2, 11665.2]
  [Round 5] Val NASA per client: [1445.8, 5011.0, 2894.1, 9329.7]
  [Round 6] 

In [10]:
from run_experiment import run_acpfl
results = run_acpfl('FD001', 42, num_clusters=2, share_heads=True)
print("AC-PFL SharedHeads:", results)


-- K-Means Clustering Results --
  - Cluster 0 assigned 54 engines.
  - Cluster 1 assigned 46 engines.
---------------------------------
✅ Created sequences: X shape = (3281, 30, 24), y shape = (3281,)
✅ Created sequences: X shape = (925, 30, 24), y shape = (925,)
✅ Created sequences: X shape = (3803, 30, 24), y shape = (3803,)
✅ Created sequences: X shape = (1151, 30, 24), y shape = (1151,)
✅ Created sequences: X shape = (3270, 30, 24), y shape = (3270,)
✅ Created sequences: X shape = (944, 30, 24), y shape = (944,)
✅ Created sequences: X shape = (3582, 30, 24), y shape = (3582,)
✅ Created sequences: X shape = (775, 30, 24), y shape = (775,)
  [Round 1] Val NASA per client: [4462.5, 7902.7, 2784.2, 2361.9]
  [Round 2] Val NASA per client: [3423.0, 7332.9, 1133.5, 1662.3]
  [Round 3] Val NASA per client: [3569.8, 7750.4, 1363.2, 2368.2]
  [Round 4] Val NASA per client: [3615.3, 6101.3, 1379.8, 1154.2]
  [Round 5] Re-clustered (α=1.0). Changes: {}
  [Round 5] Assignments: {'0': 0, '1':

In [11]:
from run_experiment import run_acpfl
results = run_acpfl('FD001', 101, num_clusters=2, share_heads=True)
print("AC-PFL SharedHeads:", results)


-- K-Means Clustering Results --
  - Cluster 0 assigned 54 engines.
  - Cluster 1 assigned 46 engines.
---------------------------------
✅ Created sequences: X shape = (3249, 30, 24), y shape = (3249,)
✅ Created sequences: X shape = (957, 30, 24), y shape = (957,)
✅ Created sequences: X shape = (3588, 30, 24), y shape = (3588,)
✅ Created sequences: X shape = (1366, 30, 24), y shape = (1366,)
✅ Created sequences: X shape = (3317, 30, 24), y shape = (3317,)
✅ Created sequences: X shape = (897, 30, 24), y shape = (897,)
✅ Created sequences: X shape = (3575, 30, 24), y shape = (3575,)
✅ Created sequences: X shape = (782, 30, 24), y shape = (782,)
  [Round 1] Val NASA per client: [2917.3, 9109.1, 3283.7, 1749.8]
  [Round 2] Val NASA per client: [2803.9, 5698.6, 2462.4, 2121.4]
  [Round 3] Val NASA per client: [2637.1, 6246.2, 3390.5, 2477.2]
  [Round 4] Val NASA per client: [2421.3, 6273.6, 2442.5, 2183.1]
  [Round 5] Re-clustered (α=1.0). Changes: {}
  [Round 5] Assignments: {'0': 0, '1':

In [12]:
from run_experiment import run_acpfl
results = run_acpfl('FD001', 202, num_clusters=2, share_heads=True)
print("AC-PFL SharedHeads:", results)


-- K-Means Clustering Results --
  - Cluster 0 assigned 54 engines.
  - Cluster 1 assigned 46 engines.
---------------------------------
✅ Created sequences: X shape = (3228, 30, 24), y shape = (3228,)
✅ Created sequences: X shape = (978, 30, 24), y shape = (978,)
✅ Created sequences: X shape = (3704, 30, 24), y shape = (3704,)
✅ Created sequences: X shape = (1250, 30, 24), y shape = (1250,)
✅ Created sequences: X shape = (3407, 30, 24), y shape = (3407,)
✅ Created sequences: X shape = (807, 30, 24), y shape = (807,)
✅ Created sequences: X shape = (3378, 30, 24), y shape = (3378,)
✅ Created sequences: X shape = (979, 30, 24), y shape = (979,)
  [Round 1] Val NASA per client: [2413.1, 13620.6, 2393.0, 7984.7]
  [Round 2] Val NASA per client: [2360.3, 11264.2, 2252.9, 6820.4]
  [Round 3] Val NASA per client: [2044.8, 7119.7, 1951.9, 6140.2]
  [Round 4] Val NASA per client: [1804.1, 8436.0, 1864.2, 4544.6]
  [Round 5] Re-clustered (α=1.0). Changes: {}
  [Round 5] Assignments: {'0': 0, '1

In [6]:
from run_experiment import run_acpfl
results = run_acpfl('FD001', 303, num_clusters=2, share_heads=True)
print("AC-PFL SharedHeads:", results)

E0000 00:00:1786429698.968016     115 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1786429699.022822     115 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1786429699.481094     115 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1786429699.481139     115 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1786429699.481142     115 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1786429699.481145     115 computation_placer.cc:177] computation placer already registered. Please check linka


-- K-Means Clustering Results --
  - Cluster 0 assigned 54 engines.
  - Cluster 1 assigned 46 engines.
---------------------------------
✅ Created sequences: X shape = (3238, 30, 24), y shape = (3238,)
✅ Created sequences: X shape = (968, 30, 24), y shape = (968,)
✅ Created sequences: X shape = (3950, 30, 24), y shape = (3950,)
✅ Created sequences: X shape = (1004, 30, 24), y shape = (1004,)
✅ Created sequences: X shape = (3321, 30, 24), y shape = (3321,)
✅ Created sequences: X shape = (893, 30, 24), y shape = (893,)
✅ Created sequences: X shape = (3460, 30, 24), y shape = (3460,)
✅ Created sequences: X shape = (897, 30, 24), y shape = (897,)


I0000 00:00:1786429730.190218     115 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1786429730.196182     115 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1786429737.257671     166 cuda_dnn.cc:529] Loaded cuDNN version 91002


  [Round 1] Val NASA per client: [3775.3, 6955.1, 2696.0, 3315.2]
  [Round 2] Val NASA per client: [3488.7, 6137.4, 2548.5, 2413.8]
  [Round 3] Val NASA per client: [3276.0, 5288.3, 2134.1, 1846.2]
  [Round 4] Val NASA per client: [4445.9, 5251.6, 1963.3, 1577.0]
  ⚠️ Spectral produced invalid clusters {0: 3, 1: 1}. Falling back to KMeans.
  KMeans fallback result: {1: 3, 0: 1}
  [Round 5] Re-clustered (α=1.0). Changes: {0: (0, 1)}
  [Round 5] Assignments: {'0': 1, '1': 0, '2': 1, '3': 1}
  [Round 5] Cluster 0 val NASA: 7505.11
  [Round 5] Cluster 1 val NASA: 2316.52
  [Round 5] Val NASA per client: [3470.0, 7505.1, 1885.4, 1594.2]
  [Round 6] Val NASA per client: [2931.7, 5908.2, 2266.6, 2101.4]
  [Round 7] Val NASA per client: [2936.9, 8167.3, 2347.5, 2939.9]
  [Round 8] Val NASA per client: [3754.8, 7192.4, 1985.0, 1380.1]
  [Round 9] Val NASA per client: [2289.5, 7324.2, 2182.7, 1826.2]
  [Round 10] Re-clustered (α=1.0). Changes: {0: (1, 0), 1: (0, 1), 2: (1, 0)}
  [Round 10] Assig

In [6]:
from run_experiment import run_acpfl
results = run_acpfl('FD001', 2026, num_clusters=2, share_heads=True)
print("AC-PFL SharedHeads:", results)

E0000 00:00:1786422236.348166     115 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1786422236.399849     115 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1786422236.848188     115 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1786422236.848222     115 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1786422236.848225     115 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1786422236.848227     115 computation_placer.cc:177] computation placer already registered. Please check linka


-- K-Means Clustering Results --
  - Cluster 0 assigned 54 engines.
  - Cluster 1 assigned 46 engines.
---------------------------------
✅ Created sequences: X shape = (3350, 30, 24), y shape = (3350,)
✅ Created sequences: X shape = (856, 30, 24), y shape = (856,)
✅ Created sequences: X shape = (3729, 30, 24), y shape = (3729,)
✅ Created sequences: X shape = (1225, 30, 24), y shape = (1225,)
✅ Created sequences: X shape = (3346, 30, 24), y shape = (3346,)
✅ Created sequences: X shape = (868, 30, 24), y shape = (868,)
✅ Created sequences: X shape = (3312, 30, 24), y shape = (3312,)
✅ Created sequences: X shape = (1045, 30, 24), y shape = (1045,)


I0000 00:00:1786422268.210627     115 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1786422268.217080     115 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1786422275.249531     167 cuda_dnn.cc:529] Loaded cuDNN version 91002


  [Round 1] Val NASA per client: [1813.5, 6135.6, 3203.1, 9951.8]
  [Round 2] Val NASA per client: [1664.6, 5133.7, 3100.2, 7216.6]
  [Round 3] Val NASA per client: [2900.2, 4664.4, 2958.1, 7556.9]
  [Round 4] Val NASA per client: [1528.9, 5559.0, 2630.6, 5106.9]
  ⚠️ Spectral produced invalid clusters {0: 1, 1: 3}. Falling back to KMeans.
  KMeans fallback result: {1: 1, 0: 3}
  [Round 5] Re-clustered (α=1.0). Changes: {0: (0, 1), 2: (1, 0), 3: (1, 0)}
  [Round 5] Assignments: {'0': 1, '1': 0, '2': 0, '3': 0}
  [Round 5] Cluster 0 val NASA: 5570.15
  [Round 5] Cluster 1 val NASA: 2183.58
  [Round 5] Val NASA per client: [2183.6, 6646.4, 2860.3, 7203.7]
  [Round 6] Val NASA per client: [2404.3, 4971.5, 2236.2, 7915.9]
  [Round 7] Val NASA per client: [1756.4, 5110.8, 2332.4, 4584.6]
  [Round 8] Val NASA per client: [1415.8, 7484.7, 2492.8, 7096.5]
  [Round 9] Val NASA per client: [2475.8, 7238.5, 2939.3, 4961.0]
  ⚠️ Spectral produced invalid clusters {1: 3, 0: 1}. Falling back to KMea

In [6]:
# AC-PFL — FD004 subset (C=3, 2 fault modes)
from run_experiment import run_acpfl, run_simulation
result = run_acpfl('FD004', 42, alpha=1.0,
                   condition_subset={0, 1, 2},
                   result_label='FD004_C3')
print("AC-PFL FD004_C3 seed 42:", result)

E0000 00:00:1787552709.701183     112 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1787552709.821646     112 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1787552710.843109     112 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1787552710.843155     112 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1787552710.843158     112 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1787552710.843160     112 computation_placer.cc:177] computation placer already registered. Please check linka

  -- Condition-regime clustering: silhouette = 0.997 --
  ⚠️ 249 engine(s) span >1 condition regime; majority-vote label applied: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10] ...


ValueError: condition_subset={0, 1, 2} matched zero engines out of 249. Check condition IDs are in range [0, n_conditions).

In [ ]:
# AC-PFL — FD004 subset (C=3, 2 fault modes)
from run_experiment import run_acpfl, run_simulation
result = run_acpfl('FD004', 101, alpha=1.0,
                   condition_subset={0, 1, 2},
                   result_label='FD004_C3')
print("AC-PFL FD004_C3 seed 101:", result)

In [ ]:
# AC-PFL — FD004 subset (C=3, 2 fault modes)
from run_experiment import run_acpfl, run_simulation
result = run_acpfl('FD004', 202, alpha=1.0,
                   condition_subset={0, 1, 2},
                   result_label='FD004_C3')
print("AC-PFL FD004_C3 seed 202:", result)

In [ ]:
# AC-PFL — FD004 subset (C=3, 2 fault modes)
from run_experiment import run_acpfl, run_simulation
result = run_acpfl('FD004', 303, alpha=1.0,
                   condition_subset={0, 1, 2},
                   result_label='FD004_C3')
print("AC-PFL FD004_C3 seed 303:", result)

In [ ]:
# AC-PFL — FD004 subset (C=3, 2 fault modes)
from run_experiment import run_acpfl, run_simulation
result = run_acpfl('FD004', 2026, alpha=1.0,
                   condition_subset={0, 1, 2},
                   result_label='FD004_C3')
print("AC-PFL FD004_C3 seed 2026:", result)

In [8]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from collections import Counter

DATA_DIR = '/kaggle/input/datasets/fardinkaiser/thesis-dataset'

df = pd.read_csv(f'{DATA_DIR}/train_FD004.txt', sep=r'\s+', header=None,
                 names=['unit_id','cycle','setting1','setting2','setting3'] +
                       [f's{i}' for i in range(1,22)])
df = df.dropna(axis=1, how='all')

setting_cols = ['setting1', 'setting2', 'setting3']
scaler = StandardScaler()
X = scaler.fit_transform(df[setting_cols])

km = KMeans(n_clusters=6, random_state=42, n_init=10)
raw_labels = km.fit_predict(X)

sil = silhouette_score(X, raw_labels)
print(f"Silhouette: {sil:.3f}")
print(f"Raw label distribution: {dict(Counter(raw_labels))}")

# Remap by setting1 centroid order
centroid_order = np.argsort(km.cluster_centers_[:, 0])
remap = {int(old): new for new, old in enumerate(centroid_order)}
print(f"Remap: {remap}")

labels = np.array([remap[c] for c in raw_labels])
print(f"Remapped label distribution: {dict(Counter(labels))}")

# Majority-vote per engine
tmp = df[['unit_id']].copy()
tmp['_cond'] = labels

unit_to_condition = {}
for uid, grp in tmp.groupby('unit_id'):
    counts = grp['_cond'].value_counts()
    unit_to_condition[int(uid)] = int(counts.idxmax())

final_dist = Counter(unit_to_condition.values())
print(f"Majority-vote condition distribution across 249 engines: {dict(sorted(final_dist.items()))}")
print(f"Engines in conditions {{0,1,2}}: {sum(v for k,v in final_dist.items() if k in {0,1,2})}")
print(f"Engines in conditions {{3,4,5}}: {sum(v for k,v in final_dist.items() if k in {3,4,5})}")

Silhouette: 0.997
Raw label distribution: {0: 15395, 3: 9091, 2: 9139, 5: 9162, 4: 9238, 1: 9224}
Remap: {4: 0, 1: 1, 3: 2, 2: 3, 5: 4, 0: 5}
Remapped label distribution: {5: 15395, 2: 9091, 3: 9139, 4: 9162, 0: 9238, 1: 9224}
Majority-vote condition distribution across 249 engines: {5: 249}
Engines in conditions {0,1,2}: 0
Engines in conditions {3,4,5}: 249


In [9]:
# Run this on Kaggle -- same diagnostic for FD002

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from collections import Counter

DATA_DIR = '/kaggle/input/datasets/fardinkaiser/thesis-dataset'

df = pd.read_csv(f'{DATA_DIR}/train_FD002.txt', sep=r'\s+', header=None,
                 names=['unit_id','cycle','setting1','setting2','setting3'] +
                       [f's{i}' for i in range(1,22)])
df = df.dropna(axis=1, how='all')

setting_cols = ['setting1', 'setting2', 'setting3']
scaler = StandardScaler()
X = scaler.fit_transform(df[setting_cols])

km = KMeans(n_clusters=6, random_state=42, n_init=10)
raw_labels = km.fit_predict(X)

sil = silhouette_score(X, raw_labels)
print(f"Silhouette: {sil:.3f}")
print(f"Raw label distribution: {dict(Counter(raw_labels))}")

centroid_order = np.argsort(km.cluster_centers_[:, 0])
remap = {int(old): new for new, old in enumerate(centroid_order)}
print(f"Remap: {remap}")

labels = np.array([remap[c] for c in raw_labels])
print(f"Remapped label distribution: {dict(Counter(labels))}")

tmp = df[['unit_id']].copy()
tmp['_cond'] = labels

unit_to_condition = {}
for uid, grp in tmp.groupby('unit_id'):
    counts = grp['_cond'].value_counts()
    unit_to_condition[int(uid)] = int(counts.idxmax())

final_dist = Counter(unit_to_condition.values())
print(f"Majority-vote condition distribution across engines: {dict(sorted(final_dist.items()))}")
print(f"Engines in conditions {{0,1,2}}: {sum(v for k,v in final_dist.items() if k in {{0,1,2}})}")
print(f"Engines in conditions {{3,4,5}}: {sum(v for k,v in final_dist.items() if k in {{3,4,5}})}")

Silhouette: 0.997
Raw label distribution: {5: 8037, 0: 13458, 3: 8002, 2: 8122, 1: 8044, 4: 8096}
Remap: {1: 0, 4: 1, 2: 2, 3: 3, 5: 4, 0: 5}
Remapped label distribution: {4: 8037, 5: 13458, 3: 8002, 2: 8122, 0: 8044, 1: 8096}
Majority-vote condition distribution across engines: {0: 1, 3: 2, 5: 257}


TypeError: unhashable type: 'set'